# 🎯 29 · Gaussian processes, Bayesian optimisation and active learning
### A posterior over functions with calibrated uncertainty, acquisition functions that decide which experiment to run next, batches for 96-well plates, machine-learning-guided protein engineering, and active learning on a labelling budget

**BiologyToML** · Level 2 · Research frontier · Part F — Modern machine-learning building blocks · Module 29 of 44

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/ml/tutorial/BiologyToML/notebooks/29_gaussian_processes_bayesian_optimization.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 2 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, SciPy, pandas, scikit-learn, Matplotlib |
| **Prerequisites** | Level 1 (especially Modules 06 linear regression, 07 ridge, 11 kernels/SVMs, 21 Bayesian inference and MCMC, 23 model selection); comfortable with NumPy, a little SciPy, and the multivariate Gaussian |
| **Estimated time** | 180 min (reading + running + the ✏️ exercises) |
| **Builds on** | [06 · Linear regression](06_linear_regression_gwas.ipynb), [07 · Ridge and lasso](07_ridge_lasso_sparse_regression.ipynb), [11 · Support vector machines (kernels)](11_support_vector_machines.ipynb), [21 · Bayesian learning and MCMC](21_bayesian_learning_mcmc.ipynb), [23 · Model selection](23_model_selection.ipynb) |
| **Research frontier covered** | Gaussian-process regression, Bayesian optimisation of experiments, machine-learning-guided protein engineering, active learning and self-driving labs — see [`FRONTIER_MAP.md`](../FRONTIER_MAP.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> All data are **SIMULATED** by code in this notebook, so every method can be checked against a known truth: a smooth
> `enzyme_activity(T)` curve and a 2-D `culture_yield(temperature, pH)` surface with a hidden optimum (defined in the
> notebook); a four-site protein fitness landscape built from the toolkit's Potts model (`simulate_helix_bundle` +
> `simulate_protein_family`, whose true fields and couplings we keep as the oracle); and single-cell profiles from
> `simulate_scrna` with known cell types. Nothing is downloaded. Real experiments are noisier, higher-dimensional and
> far more expensive — Section 9 explains what changes.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Explain** why a Gaussian process is Bayesian linear regression with infinitely many features, and **read** a kernel (RBF, Matérn, linear) as an assumption about how smooth the unknown function is.
2. **Implement** GP regression from scratch with a Cholesky factorisation — posterior mean *and* calibrated variance — and **check** that its 95 % band really covers the truth about 95 % of the time.
3. **Choose** the length scale, amplitude and noise by maximising the log marginal likelihood, **see** Occam's razor at work, and **compare** the choice with cross-validation ([Module 23](23_model_selection.ipynb)).
4. **Derive** the expected-improvement acquisition function and **run** Bayesian optimisation, beating random search and a grid at finding the best culture condition in the fewest experiments.
5. **Select batches** of experiments for a 96-well plate (Thompson sampling and the constant-liar trick) and weigh rounds against total cost.
6. **Optimise a protein** over a 160,000-variant four-site library with rounds of 96 — beating random mutagenesis from 0.3 % of the library — and **compare** an epistatic (RBF) with an additive (linear) surrogate to see why the additive baseline is so hard to beat.
7. **Run active learning** on a labelling budget: query the cells the model is least sure about, and **find** the setting in which uncertainty sampling backfires.
8. **Name** the production tools (GPyTorch, BoTorch, scikit-learn GPs), the public landscapes, and the open problems of machine-learning-guided experimentation.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, course toolkit | 5 min |
| 1 · From weights to functions | Bayesian linear regression → GP; kernels; prior samples | 25 min |
| 2 · GP regression from scratch | Cholesky posterior; enzyme curve; 🧪 coverage of the band | 25 min |
| 3 · Hyper-parameters by marginal likelihood | log p(y|X,θ); Occam's razor; vs cross-validation | 20 min |
| 4 · Bayesian optimisation | PI, EI, UCB; 1-D walk-through; 🧪 2-D vs random & grid over seeds | 30 min |
| 5 · Batches of experiments | 96-well plates; Thompson sampling; constant liar; rounds vs cost | 15 min |
| 6 · Protein engineering (MLDE) | a 4-site landscape; 🧪 epistatic vs additive vs random | 25 min |
| 7 · Active learning for classification | uncertainty vs random; learning curves; when it fails | 20 min |
| 8 · Pitfalls | cubic cost, mis-specification, non-stationarity, out-of-distribution | 8 min |
| 9 · 🧭 From toy to real data | GPyTorch/BoTorch, ProteinGym/FLIP, self-driving labs | 7 min |
| ✅ / 🧠 / 🚀 / 📝 | takeaways, checkpoint, discussion, going further, lab note | 10 min |

## 🧫 The biological question

Experiments are the bottleneck of biology. Each enzyme variant has to be cloned, expressed and assayed; each culture
condition takes days in the incubator; each drug combination costs reagents and time. You can afford tens, maybe a few
hundred, measurements — not the millions a grid search over temperature × pH × media would need, and certainly not the
$20^{200}$ sequences of a protein. So the practical question is rarely "fit a model to all the data" (there is no all the
data) but:

> **Given the handful of experiments we have run so far, which experiment should we run *next* to reach the best result
> in the fewest trials — and how sure are we?**

Answering it well needs two ingredients that Level 1 only hinted at. First, a model that reports **honest uncertainty**:
not just a prediction of the yield at an untried temperature, but how much to trust it. Second, a rule that turns that
uncertainty into a **decision** — balancing *exploitation* (try near the best point so far) against *exploration* (try
where we are most ignorant, because the real optimum might hide there).

The **Gaussian process (GP)** supplies the first and **Bayesian optimisation (BO)** the second. Together they drive
machine-learning-guided protein engineering — Romero, Krause & Arnold (2013) used GPs to navigate a cytochrome P450
landscape, and the directed-evolution-meets-ML programme of Yang, Wu & Arnold (2019) built on exactly this loop — and
they are the brain of the "self-driving labs" that now plan their own experiments (King *et al.* 2009's *Robot Scientist*;
Boiko *et al.* 2023). The same machinery, used passively, becomes **active learning**: spend a scarce annotation budget on
the samples that teach the model most.

Everything here is **simulated** so we always know the right answer: a smooth enzyme-activity curve, a 2-D culture-yield
surface with a hidden optimum, a four-site protein-fitness landscape from the toolkit's Potts model, and single cells with
known types. We build the GP and the acquisition functions from scratch — they are short — and check every claim against
the ground truth.

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this module needs only a CPU.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("sklearn", "scikit-learn")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import time
from itertools import product

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.linalg import cho_factor, cho_solve, cholesky, solve_triangular
from scipy.optimize import minimize
from scipy.stats import norm
from sklearn.linear_model import LogisticRegression

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

In [ ]:
# ── 0.1 A stopwatch, to report the total compute at the end ──
T_START = time.time()
print("Everything in this notebook is CPU-only and seeded (SEED =", SEED, ").")

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py).
We use the protein-landscape simulators (`simulate_helix_bundle` for a structure and contact map, `simulate_protein_family` for a Potts model with known fields and couplings, `potts_score` to evaluate it) for the protein-engineering section, and `simulate_scrna` + `pca` + `plot_embedding` for active learning. The Gaussian process and every acquisition function are written from scratch below. Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v2.0; copied so this notebook runs alone) ──
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

MITO_GENES = ["MT-ND1", "MT-ND2", "MT-CO1", "MT-CO2", "MT-ATP8", "MT-ATP6", "MT-CO3", "MT-ND3",
              "MT-ND4L", "MT-ND4", "MT-ND5", "MT-ND6", "MT-CYB"]


INTERFERON_GENES = ["ISG15", "IFI6", "IFIT1", "IFIT3", "MX1", "OAS1", "IFI44L", "RSAD2", "IRF7", "STAT1"]


CYCLING_GENES = ["MKI67", "TOP2A", "CCNB1", "CDK1", "UBE2C", "BIRC5", "TYMS", "PCNA"]


PBMC_TYPES = {
    "CD4 T": ("lymphoid", ["CD3E", "CD3D", "IL7R", "CD4"], 0.5),
    "CD8 T": ("lymphoid", ["CD3E", "CD3D", "CD8A", "CD8B"], 0.5),
    "NK": ("lymphoid", ["NKG7", "GNLY", "KLRD1", "PRF1"], 2.2),
    "B": ("lymphoid", ["MS4A1", "CD79A", "CD79B", "CD19"], 2.4),
    "Monocyte": ("myeloid", ["CD14", "LYZ", "S100A8", "S100A9"], 2.6),
    "Dendritic": ("myeloid", ["FCER1A", "CST3", "CLEC10A", "HLA-DQA1"], 2.2),
}


def sample_negative_binomial(rng: np.random.Generator, mu, theta):
    """Negative-binomial counts with mean ``mu`` and inverse dispersion ``theta``.

    Variance = mu + mu²/theta (large theta → Poisson).  Drawn as a Gamma–Poisson mixture:
    each count is Poisson with a rate that itself varies from cell to cell.
    """
    mu = np.asarray(mu, float)
    theta = np.broadcast_to(np.asarray(theta, float), mu.shape)
    return rng.poisson(rng.gamma(theta, np.where(mu > 0, mu, 0.0) / theta)).astype(np.int32)


def make_expression_programs(rng: np.random.Generator, n_genes: int, cell_types: dict,
                             n_extra_markers: int = 20, extra_names=()):
    """Log-scale expression profiles (one row per cell type) for a simulated tissue.

    ``cell_types`` maps a type name to (lineage, named marker genes, marker strength).  Every
    type gets its named markers plus ``n_extra_markers`` anonymous ones; types of the same
    lineage share 20 lineage genes.  ``extra_names`` are reserved gene names (placed after the
    markers) that the caller gives its own behaviour.  Returns ``(log_profiles, genes, marker_of)``.
    """
    named = []
    for _, markers, _ in cell_types.values():
        named += [g for g in markers if g not in named]
    reserved = named + [g for g in extra_names if g not in named]
    if len(reserved) > n_genes:
        raise ValueError("n_genes is too small for the requested marker genes")
    genes = np.array(reserved + [f"Gene{j:04d}" for j in range(1, n_genes - len(reserved) + 1)], dtype=object)
    free = np.arange(len(reserved), n_genes)                       # anonymous genes available
    base = rng.normal(0.0, 2.0, n_genes)                           # gene-specific baseline (log scale)
    base[:len(named)] -= 2.0                                       # named markers are low elsewhere
    profiles = np.tile(base, (len(cell_types), 1)) + rng.normal(0, 0.15, (len(cell_types), n_genes))
    marker_of = np.array([""] * n_genes, dtype=object)
    lineages = sorted({v[0] for v in cell_types.values()})
    lineage_genes = {lin: rng.choice(free, 20, replace=False) for lin in lineages}
    for t, (name, (lineage, markers, strength)) in enumerate(cell_types.items()):
        profiles[t, lineage_genes[lineage]] += rng.normal(1.2, 0.3, 20)
        idx = [int(np.flatnonzero(genes == g)[0]) for g in markers]
        profiles[t, idx] += strength + 2.0 + rng.normal(0, 0.2, len(idx))
        extra = rng.choice(free, n_extra_markers, replace=False)
        profiles[t, extra] += rng.normal(0.6 + strength / 2, 0.3, n_extra_markers)
        for j in list(idx) + list(extra):
            marker_of[j] = name if marker_of[j] == "" else marker_of[j] + "/" + name
    return profiles, genes, marker_of


def simulate_scrna(rng: np.random.Generator, n_cells: int = 3000, n_genes: int = 1000, cell_types=None,
                   proportions=None, n_donors: int = 6, n_batches: int = 2, confounded: bool = False,
                   depth: float = 1500, batch_effect: float = 0.25, donor_effect: float = 0.15,
                   stim_effect: float = 1.0, theta: float = 8.0, doublet_rate: float = 0.0,
                   damaged_rate: float = 0.0, cycling_rate: float = 0.0, drop_type_in_batch=None) -> dict:
    """Simulate single-cell RNA-seq UMI counts for a PBMC-like sample collection.

    Design: ``n_donors`` donors, the first half *control* and the second half *stimulated*
    (an interferon response in every cell type plus a monocyte-specific response).  Donor
    ``d`` is sequenced in batch ``d % n_batches`` — or, with ``confounded=True``, every control
    donor in batch 1 and every stimulated donor in batch 2 (a design flaw to learn from).
    Counts are negative binomial around ``depth`` UMIs per cell.  Optional artefacts for
    quality control: doublets, damaged cells (high mitochondrial fraction), cycling cells.
    ``drop_type_in_batch=("B", 1)`` removes B cells from batch 1 (index 1, i.e. "batch2").

    Expression probabilities are renormalised within each cell (a fixed-size library, as in
    sequencing), so in stimulated cells the genes with ``true_logfc == 0`` lose a few per cent of
    their share (about −0.04 on the natural-log scale) to the induced genes: a compositional
    effect that is also present in real data.

    Returns a dict with ``counts`` (cells × genes, int32), ``obs`` (per-cell table with the
    true cell type, donor, batch, condition, …), ``var`` (per-gene table), ``true_logfc``
    (cell type × gene stimulation effect, natural log) and ``log_profiles``.
    """
    cell_types = PBMC_TYPES if cell_types is None else cell_types
    names = list(cell_types)
    T = len(names)
    p = np.full(T, 1 / T) if proportions is None else np.asarray(proportions, float) / np.sum(proportions)
    extra = MITO_GENES + INTERFERON_GENES + CYCLING_GENES
    profiles, genes, marker_of = make_expression_programs(rng, n_genes, cell_types, extra_names=extra)
    G = len(genes)
    is_mito, is_ifn, is_cyc = (np.isin(genes, MITO_GENES), np.isin(genes, INTERFERON_GENES),
                               np.isin(genes, CYCLING_GENES))
    profiles[:, is_mito] = rng.normal(3.0, 0.3, is_mito.sum())          # ≈ 4 % of UMIs are mitochondrial
    profiles[:, is_ifn | is_cyc] = rng.normal(-1.0, 0.4, (T, (is_ifn | is_cyc).sum()))
    # stimulation: interferon genes up everywhere + 40 monocyte-specific responders (up or down)
    true_logfc = np.zeros((T, G))
    true_logfc[:, is_ifn] = stim_effect * rng.normal(1.6, 0.3, is_ifn.sum())
    if "Monocyte" in names:
        responders = rng.choice(np.flatnonzero((marker_of == "") & ~is_mito & ~is_ifn & ~is_cyc), 40, replace=False)
        true_logfc[names.index("Monocyte"), responders] = stim_effect * rng.choice([-1, 1], 40) * rng.normal(1.2, 0.3, 40)
    # donors, batches, conditions
    donor = rng.integers(0, n_donors, n_cells)
    stimulated = donor >= n_donors // 2
    batch = stimulated.astype(int) if confounded else donor % n_batches
    batch = np.minimum(batch, n_batches - 1)
    cell_type = rng.choice(T, n_cells, p=p)
    if drop_type_in_batch is not None:
        t_drop, b_drop = names.index(drop_type_in_batch[0]), drop_type_in_batch[1]
        bad = (cell_type == t_drop) & (batch == b_drop)
        cell_type[bad] = rng.choice(np.delete(np.arange(T), t_drop), bad.sum(), p=np.delete(p, t_drop) / (1 - p[t_drop]))
    batch_shift = rng.normal(0, batch_effect, (n_batches, G))
    batch_depth = np.exp(rng.normal(0, 0.3, n_batches))
    donor_shift = rng.normal(0, donor_effect, (n_donors, G))
    log_rate = (profiles[cell_type] + batch_shift[batch] + donor_shift[donor]
                + true_logfc[cell_type] * stimulated[:, None])
    cycling = rng.random(n_cells) < cycling_rate
    log_rate[np.ix_(cycling, np.flatnonzero(is_cyc))] += 3.0
    damaged = rng.random(n_cells) < damaged_rate
    log_rate[np.ix_(damaged, np.flatnonzero(is_mito))] += 2.2
    lib = depth * batch_depth[batch] * np.exp(rng.normal(0, 0.35, n_cells))
    lib[damaged] *= 0.35
    prob = np.exp(log_rate - log_rate.max(axis=1, keepdims=True))
    prob /= prob.sum(axis=1, keepdims=True)
    theta_g = theta * np.exp(rng.normal(0, 0.5, G))
    mu = lib[:, None] * prob
    doublet = rng.random(n_cells) < doublet_rate
    partner = rng.integers(0, n_cells, n_cells)
    mu[doublet] = 0.9 * mu[doublet] + 0.9 * mu[partner[doublet]]       # two cells in one droplet
    counts = sample_negative_binomial(rng, mu, theta_g)
    obs = pd.DataFrame({
        "cell_type": np.where(doublet, "Doublet", np.array(names, dtype=object)[cell_type]),
        "true_type": np.array(names, dtype=object)[cell_type],
        "lineage": np.array([cell_types[n][0] for n in names], dtype=object)[cell_type],
        "donor": [f"donor{d + 1}" for d in donor],
        "condition": np.where(stimulated, "stimulated", "control"),
        "batch": [f"batch{b + 1}" for b in batch],
        "n_counts": counts.sum(axis=1),
        "n_genes": (counts > 0).sum(axis=1),
        "pct_mito": 100 * counts[:, is_mito].sum(axis=1) / np.maximum(counts.sum(axis=1), 1),
        "is_doublet": doublet, "is_damaged": damaged, "is_cycling": cycling,
    }, index=[f"cell{i:05d}" for i in range(n_cells)])
    var = pd.DataFrame({"marker_of": marker_of, "is_mito": is_mito, "is_interferon": is_ifn,
                        "is_cycling": is_cyc, "theta": theta_g}, index=genes)
    return dict(counts=counts, obs=obs, var=var, log_profiles=pd.DataFrame(profiles, index=names, columns=genes),
                true_logfc=pd.DataFrame(true_logfc, index=names, columns=genes))


AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"


def simulate_helix_bundle(n_helices: int = 4, helix_length: int = 14, loop_length: int = 4,
                          rng=None, noise: float = 0.0) -> dict:
    """Cα coordinates (Å) of an idealised four-helix-bundle protein, its contact map and secondary structure.

    Helices are ideal right-handed α-helices (3.6 residues per turn, 1.5 Å rise, 2.3 Å radius)
    packed antiparallel on the corners of a 10 Å square and joined by short loops.  ``noise`` adds
    Gaussian jitter (needs ``rng``).  Contacts: Cα–Cα distance < 8 Å and |i − j| ≥ 3.

    Returns a dict with ``coords`` (L × 3), ``ss`` (string of 'H'/'L'), ``distances`` and ``contacts``.
    """
    corners = 5.0 * np.array([[1, 1], [1, -1], [-1, -1], [-1, 1]], float)
    pts, ss = [], ""
    for h in range(n_helices):
        cx, cy = corners[h % 4] * (1 + h // 4)
        up = h % 2 == 0
        k = np.arange(helix_length)
        angle = np.deg2rad(100.0) * k * (1 if up else -1)      # turn with the direction of travel, so
        z = 1.5 * (k - (helix_length - 1) / 2) * (1 if up else -1)   # every helix is right-handed
        helix = np.c_[cx + 2.3 * np.cos(angle), cy + 2.3 * np.sin(angle), z]
        if pts:                                                      # loop: circular arc, 3.8 Å steps
            start, end = pts[-1], helix[0]
            d, m = np.linalg.norm(end - start), loop_length + 1
            lo, hi = 1e-6, 2 * np.pi - 1e-6                              # arc angle Θ with m equal chords
            for _ in range(60):
                theta = (lo + hi) / 2
                lo, hi = (theta, hi) if 3.8 * np.sin(theta / 2) / np.sin(theta / (2 * m)) > d else (lo, theta)
            R = 1.9 / np.sin(theta / (2 * m))
            e1 = (end - start) / d
            e2 = np.r_[0, 0, np.sign(start[2])] - np.dot(np.r_[0, 0, np.sign(start[2])], e1) * e1
            e2 /= np.linalg.norm(e2)
            c = (start + end) / 2 - R * np.cos(theta / 2) * e2
            for k in range(1, m):
                psi = -theta / 2 + k * theta / m
                pts.append(c + R * (np.sin(psi) * e1 + np.cos(psi) * e2))
            ss += "L" * loop_length
        pts.extend(helix)
        ss += "H" * helix_length
    coords = np.array(pts)
    if noise > 0:
        coords = coords + rng.normal(0, noise, coords.shape)
    D = np.linalg.norm(coords[:, None] - coords[None], axis=-1)
    sep = np.abs(np.arange(len(coords))[:, None] - np.arange(len(coords))[None])
    return dict(coords=coords, ss=ss, distances=D, contacts=(D < 8.0) & (sep >= 3))


def potts_score(seqs, h, J) -> np.ndarray:
    """Statistical fitness Σ_i h_i(a_i) + Σ_{i<j} J_ij(a_i, a_j) of integer-coded sequences (higher = fitter)."""
    seqs = np.atleast_2d(seqs)
    L = seqs.shape[1]
    ii, jj = np.nonzero(np.triu(np.abs(J).sum(axis=(2, 3)) > 0, 1))      # coupled pairs only
    return h[np.arange(L), seqs].sum(axis=1) + J[ii, jj, seqs[:, ii], seqs[:, jj]].sum(axis=1)


def simulate_protein_family(rng: np.random.Generator, contacts: np.ndarray, n_seqs: int = 3000,
                            q: int = 20, field_scale: float = 1.0, coupling_scale: float = 1.0,
                            n_sweeps: int = 100) -> dict:
    """Simulate a protein family (a multiple sequence alignment) from a Potts model on a contact map.

    Each residue pair in contact gets a q × q coupling matrix J_ij (pairs that fit together
    are rewarded), each position a field h_i (its conservation).  Sequences are drawn by
    Gibbs sampling (``n_seqs`` independent chains, ``n_sweeps`` sweeps each), so conserved
    positions and *co-varying* contact pairs appear exactly as in natural alignments — but
    without phylogeny.  The wild type is the fittest sampled sequence.

    Returns a dict with ``msa`` (n_seqs × L integers 0…q−1; letters = AMINO_ACIDS[:q]),
    ``h`` (L × q), ``J`` (L × L × q × q, symmetric: J[j, i] = J[i, j].T), ``wild_type``.
    """
    L = contacts.shape[0]
    cons = rng.uniform(0.3, 1.5, L)[:, None] * field_scale
    h = rng.normal(0, 1, (L, q)) * cons
    J = np.zeros((L, L, q, q))
    pairs = [(i, j) for i in range(L) for j in range(i + 1, L) if contacts[i, j]]
    for i, j in pairs:
        J[i, j] = rng.normal(0, 0.5 * coupling_scale, (q, q))
        J[j, i] = J[i, j].T
    nbrs = [np.flatnonzero(contacts[i]) for i in range(L)]
    x = rng.integers(0, q, (n_seqs, L))
    rows = np.arange(n_seqs)
    for _ in range(n_sweeps):
        for i in rng.permutation(L):
            logits = np.tile(h[i], (n_seqs, 1))
            for j in nbrs[i]:
                logits += J[j, i][x[:, j]]                                 # = J_ij(·, x_j)
            g = logits - np.log(-np.log(rng.random((n_seqs, q))))           # Gumbel-max sampling
            x[rows, i] = np.argmax(g, axis=1)
    wild_type = x[np.argmax(potts_score(x, h, J))].copy()
    return dict(msa=x, h=h, J=J, wild_type=wild_type)


def standardize(X, mean=None, sd=None):
    """Z-score the columns of X. Pass the training mean/sd to transform test data the same way.

    Returns ``(Z, mean, sd)``; constant columns get sd = 1 so they map to 0.
    """
    X = np.asarray(X, float)
    mean = X.mean(axis=0) if mean is None else mean
    sd = X.std(axis=0) if sd is None else sd
    sd = np.where(sd > 0, sd, 1.0)
    return (X - mean) / sd, mean, sd


def log_normalize(counts, target_sum: float = 1e4):
    """Library-size normalisation + log: log(1 + target_sum · count / total count of the cell)."""
    counts = np.asarray(counts, float)
    total = counts.sum(axis=1, keepdims=True)
    return np.log1p(target_sum * counts / np.where(total > 0, total, 1.0))


def pca(X, n_components: int = 2):
    """Principal component analysis by SVD of the column-centred data.

    Returns ``(scores, components, explained_variance_ratio)``: scores are N × k, components k × J.
    """
    X = np.asarray(X, float)
    Xc = X - X.mean(axis=0)
    U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
    var = S ** 2 / (len(X) - 1)
    return U[:, :n_components] * S[:n_components], Vt[:n_components], var[:n_components] / var.sum()


CLASS_COLORS = ["#1f77b4", "#d62728", "#2ca02c", "#9467bd", "#ff7f0e", "#8c564b"]


def plot_embedding(Z, labels, ax=None, title: str = "", s: float = 4, legend: bool = True, cmap=None):
    """Scatter a 2-D embedding (PCA, t-SNE, latent space, …) coloured by a categorical or numeric label."""
    Z = np.asarray(Z, float)
    labels = np.asarray(labels)
    ax = ax or plt.gca()
    if labels.dtype.kind in "fc":
        sc = ax.scatter(Z[:, 0], Z[:, 1], c=labels, s=s, cmap=cmap or "viridis", linewidths=0)
        plt.colorbar(sc, ax=ax, shrink=0.8)
    else:
        palette = plt.get_cmap("tab10" if len(np.unique(labels)) <= 10 else "tab20")
        for k, lab in enumerate(np.unique(labels)):
            m = labels == lab
            ax.scatter(Z[m, 0], Z[m, 1], s=s, color=palette(k % palette.N), label=str(lab), linewidths=0)
        if legend:
            ax.legend(markerscale=3, fontsize=7, frameon=False, loc="best")
    ax.set_xticks([]); ax.set_yticks([])
    if title:
        ax.set_title(title)
    return ax


print("🧰 Toolkit loaded:", ", ".join(['sample_negative_binomial', 'make_expression_programs', 'simulate_scrna', 'simulate_helix_bundle', 'potts_score', 'simulate_protein_family', 'standardize', 'log_normalize', 'pca', 'plot_embedding']))

---
## 1 · From weights over features to a distribution over functions

> 📍 **Where we are.** [Module 06](06_linear_regression_gwas.ipynb) fit a line by least squares; [Module 07](07_ridge_lasso_sparse_regression.ipynb)
> added a Gaussian prior on the weights (ridge = MAP) and [Module 21](21_bayesian_learning_mcmc.ipynb) kept the *whole*
> posterior instead of a point estimate. A Gaussian process is the natural next step: keep the Bayesian posterior, let the
> number of features go to infinity, and work with the functions directly.

### 🧮 Math refresher — Bayesian linear regression, in one screen

Model the data as $y = \phi(x)^\top w + \varepsilon$ with features $\phi(x)\in\mathbb{R}^M$, a Gaussian prior
$w\sim\mathcal N(0,\,\Sigma_p)$ and noise $\varepsilon\sim\mathcal N(0,\sigma^2)$. Given a training matrix $\Phi$
($n\times M$, row $i$ is $\phi(x_i)^\top$) and targets $y$, the posterior over weights is Gaussian (this is just ridge,
Module 07, kept as a distribution):

$$p(w\mid X,y)=\mathcal N\!\big(\bar w,\,A^{-1}\big),\qquad A=\sigma^{-2}\Phi^\top\Phi+\Sigma_p^{-1},\quad
\bar w=\sigma^{-2}A^{-1}\Phi^\top y.$$

*In words:* seeing data **sharpens** the prior into a narrower Gaussian centred near the ridge solution. The prediction at
a new point $x_\star$ is also Gaussian, $f_\star=\phi(x_\star)^\top w$, with mean $\phi_\star^\top\bar w$ and a variance
that **grows as $x_\star$ moves away from the data**. That growing variance — "I have not measured anything nearby, so I
am unsure" — is the whole point.

Now the trick. Everywhere the predictive distribution uses the features, they appear only inside inner products
$\phi(x)^\top\Sigma_p\,\phi(x')$. Define the **kernel** $k(x,x')=\phi(x)^\top\Sigma_p\,\phi(x')$ and you never need the
features themselves — exactly the kernel trick of [Module 11](11_support_vector_machines.ipynb). A **Gaussian process** is
what you get in the limit of infinitely many features: a prior *directly on functions*, such that the function values at
any finite set of points are jointly Gaussian,

$$f \sim \mathcal{GP}\big(m(\cdot),\,k(\cdot,\cdot)\big),\qquad
\begin{bmatrix}f(x_1)\\\vdots\\f(x_n)\end{bmatrix}\sim\mathcal N\big(m,\,K\big),\;\; K_{ij}=k(x_i,x_j).$$

*In words:* the **mean function** $m$ (we take $m=0$ after centring) is our best guess before data; the **kernel** $K$
says how strongly two points' values co-vary — points the kernel calls "close" are forced to have similar $f$. Choosing a
kernel *is* choosing what we believe about the unknown function.

> 🔗 **Builds on.** The posterior-sharpens-the-prior picture is [Module 21 §Bayesian inference](21_bayesian_learning_mcmc.ipynb);
> the $\Sigma_p=\tau^2 I$ special case is ridge regression ([Module 07](07_ridge_lasso_sparse_regression.ipynb)); the
> "inner products only" move is the kernel trick from [Module 11](11_support_vector_machines.ipynb). A GP just takes all
> three to their Bayesian limit.

In [ ]:
# ── 1.1 Three kernels: how "nearby" is defined ──
def rbf_kernel(X1, X2, length=1.0, signal=1.0):
    """Squared-exponential (RBF) kernel: k = signal² · exp(−‖x−x'‖²/(2·length²)). Infinitely smooth functions."""
    X1, X2 = np.atleast_2d(X1), np.atleast_2d(X2)
    d2 = np.sum(X1**2, 1)[:, None] + np.sum(X2**2, 1)[None, :] - 2 * X1 @ X2.T
    return signal**2 * np.exp(-np.maximum(d2, 0) / (2 * length**2))


def matern32_kernel(X1, X2, length=1.0, signal=1.0):
    """Matérn-3/2 kernel: k = signal²·(1+√3 r/ℓ)·exp(−√3 r/ℓ). Rougher, once-differentiable functions."""
    X1, X2 = np.atleast_2d(X1), np.atleast_2d(X2)
    d2 = np.sum(X1**2, 1)[:, None] + np.sum(X2**2, 1)[None, :] - 2 * X1 @ X2.T
    r = np.sqrt(np.maximum(d2, 0)) * np.sqrt(3) / length
    return signal**2 * (1 + r) * np.exp(-r)


def linear_kernel(X1, X2, signal=1.0, bias=0.0):
    """Linear kernel: k = signal²·x·x' + bias. The GP is then ordinary Bayesian linear regression (straight lines)."""
    return signal**2 * np.atleast_2d(X1) @ np.atleast_2d(X2).T + bias


print("Three kernels defined. The RBF length scale ℓ sets how far apart two inputs must be before the")
print("function is allowed to change — biologically, 'over how many °C does activity vary independently?'")

In [ ]:
# ── 1.2 Draw functions from the GP *prior* (before any data) ──
rng = np.random.default_rng(SEED + 1)
grid = np.linspace(0, 10, 200)[:, None]
fig, axes = plt.subplots(1, 3, figsize=(11, 3.2), sharey=True)
for ax, length in zip(axes, [0.5, 1.5, 4.0]):
    K = rbf_kernel(grid, grid, length=length, signal=1.0) + 1e-9 * np.eye(len(grid))
    Lc = cholesky(K, lower=True)
    for _ in range(5):
        f = Lc @ rng.standard_normal(len(grid))                 # sample = L · standard normal
        ax.plot(grid.ravel(), f, lw=1.1)
    ax.axhspan(-2, 2, color="0.85", alpha=0.4, zorder=0)         # ±2 = 95 % prior band (signal = 1)
    ax.set_title(f"RBF prior, length ℓ = {length}"); ax.set_xlabel("input x")
axes[0].set_ylabel("function value f(x)")
fig.suptitle("SIMULATED draws from a GP prior — short ℓ = wiggly, long ℓ = smooth", y=1.03)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Each curve is a *sample function* drawn from the prior — we never specified a parametric form
(no polynomial degree, no number of basis functions), yet the GP happily produces random smooth functions. The only knob
is the **length scale** $\ell$: small $\ell$ lets the function change over a short distance (wiggly), large $\ell$ forces
it to vary slowly (smooth). The grey band is the prior 95 % interval, $\pm2$ because we set the signal amplitude to 1.
Before any data, the GP says "the function is somewhere in here, equally likely to be high or low." Section 2 conditions on
observations and watches the band collapse around them.

✏️ **Your turn.** Swap `rbf_kernel` for `matern32_kernel` in 1.2. The Matérn-3/2 samples are visibly *rougher* (continuous
but not smooth). Which is a better prior for enzyme activity vs temperature, and which for a jagged binding-affinity
landscape? (There is no single right answer — that is the modelling choice a kernel encodes.)

---
## 2 · Gaussian-process regression from scratch

> 📍 **Where we are.** We have a prior over functions. Now we **condition** on noisy measurements. Because everything is
> jointly Gaussian, the posterior is available in closed form — no MCMC, no optimisation — using one Cholesky factorisation.

### 🧮 The posterior in closed form

Observe $y_i=f(x_i)+\varepsilon_i$ with $\varepsilon_i\sim\mathcal N(0,\sigma^2)$. The training values and a test value are
jointly Gaussian, so conditioning a Gaussian on part of itself (a standard identity) gives the **predictive
distribution** at test inputs $X_\star$:

$$\boxed{\;\bar f_\star = K_\star^\top (K+\sigma^2 I)^{-1} y,\qquad
\operatorname{cov}(f_\star)=K_{\star\star}-K_\star^\top (K+\sigma^2 I)^{-1}K_\star\;}$$

where $K=k(X,X)$, $K_\star=k(X,X_\star)$ and $K_{\star\star}=k(X_\star,X_\star)$.

*In words:* the **mean** is a weighted average of the observed $y$'s, weighting training points by how much the kernel
links them to $x_\star$ — a smooth, Bayesian cousin of $k$-nearest-neighbours. The **variance** starts from the prior
$K_{\star\star}$ and *subtracts* what the data explain; it is small near observations and swells in the gaps. Crucially,
the variance does not depend on the observed $y$ values at all — only on *where* you measured.

We never invert $K+\sigma^2 I$ directly. We Cholesky-factor it once, $K+\sigma^2 I = LL^\top$, then solve triangular
systems — numerically stable and $O(n^3)$ once, $O(n^2)$ per test point. That cubic cost is the GP's Achilles heel
(Section 8).

In [ ]:
# ── 2.1 GP regression in a dozen lines ──
def gp_predict(X, y, Xs, kernel, noise, return_cov=False, jitter=1e-8):
    """Posterior mean and variance of a zero-mean GP. y is centred internally; `kernel(A, B)` returns k(A, B)."""
    X, Xs = np.atleast_2d(X), np.atleast_2d(Xs)
    y_mean = y.mean()
    yc = y - y_mean
    K = kernel(X, X) + (noise**2 + jitter) * np.eye(len(X))
    L = cholesky(K, lower=True)                                   # K = L Lᵀ (stable, done once)
    alpha = cho_solve((L, True), yc)                             # (K+σ²I)⁻¹ y
    Ks = kernel(X, Xs)
    mean = Ks.T @ alpha + y_mean
    v = solve_triangular(L, Ks, lower=True)                      # L⁻¹ Ks
    var = np.diag(kernel(Xs, Xs)) - np.sum(v**2, axis=0)         # Kss − KsᵀK⁻¹Ks (diagonal only)
    var = np.maximum(var, 1e-12)
    if return_cov:
        cov = kernel(Xs, Xs) - v.T @ v
        return mean, var, cov
    return mean, var


print("gp_predict ready: returns the posterior mean and the *latent* variance (add noise² for a measurement interval).")

In [ ]:
# ── 2.2 A SIMULATED enzyme: activity vs temperature (the hidden truth) ──
def enzyme_activity(T):
    """True (noise-free) relative activity of a thermostable enzyme vs temperature in °C. Peaks, then denatures."""
    T = np.asarray(T, float)
    return np.exp(-0.5 * ((T - 42) / 9.0) ** 2) - 0.12 * np.exp(-0.5 * ((T - 20) / 6.0) ** 2)


TRUE_NOISE = 0.08                                                # assay measurement noise (std)
rng = np.random.default_rng(SEED + 2)
T_obs = rng.uniform(10, 70, 16)[:, None]                        # 16 temperatures we have assayed
y_obs = enzyme_activity(T_obs).ravel() + rng.normal(0, TRUE_NOISE, 16)
T_grid = np.linspace(5, 75, 300)[:, None]
print("Assayed temperatures (°C):", np.round(np.sort(T_obs.ravel()), 1))
print("Peak of the TRUE curve is at T =", round(float(T_grid[np.argmax(enzyme_activity(T_grid)), 0]), 1), "°C")

In [ ]:
# ── 2.3 Fit the GP and plot the posterior over the true curve ──
kern = lambda A, B: rbf_kernel(A, B, length=7.0, signal=0.45)   # a sensible guess; Section 3 learns these
mean, var = gp_predict(T_obs, y_obs, T_grid, kern, noise=TRUE_NOISE)
sd = np.sqrt(var)

fig, ax = plt.subplots(figsize=(6.6, 3.8))
ax.plot(T_grid, enzyme_activity(T_grid), "k--", lw=1.3, label="true activity (hidden)")
ax.plot(T_grid, mean, color="tab:blue", lw=1.6, label="GP posterior mean")
ax.fill_between(T_grid.ravel(), mean - 1.96 * sd, mean + 1.96 * sd, color="tab:blue", alpha=0.2,
                label="95 % band (latent f)")
ax.scatter(T_obs, y_obs, c="tab:red", zorder=5, label="assays (SIMULATED)")
ax.set_xlabel("temperature (°C)"); ax.set_ylabel("relative activity")
ax.set_title("GP regression: mean + honest uncertainty, wide where we have not measured")
ax.legend(fontsize=7.5); plt.tight_layout(); plt.show()

🔍 **What just happened?** The posterior mean threads through the assays and follows the hidden curve closely where we
have data; between and beyond the measured temperatures the band **fans out** — the GP tells us exactly where it is
guessing. This is the property no point-estimate model from Level 1 had: ridge or a random forest would draw a line with
no sense of "but I have never seen 60 °C." That calibrated ignorance is what Bayesian optimisation will exploit.

In [ ]:
# ── 2.4 🧪 Ground-truth check: does the 95 % band actually cover 95 %? ──
rng = np.random.default_rng(SEED + 3)
T_test = rng.uniform(10, 70, 2000)[:, None]
y_test = enzyme_activity(T_test).ravel() + rng.normal(0, TRUE_NOISE, 2000)   # fresh noisy measurements
m_t, v_t = gp_predict(T_obs, y_obs, T_test, kern, noise=TRUE_NOISE)
sd_latent = np.sqrt(v_t)
sd_pred = np.sqrt(v_t + TRUE_NOISE**2)                          # add measurement noise for a *measurement* interval
cover_latent = np.mean(np.abs(enzyme_activity(T_test).ravel() - m_t) <= 1.96 * sd_latent)
cover_pred = np.mean(np.abs(y_test - m_t) <= 1.96 * sd_pred)
rmse = np.sqrt(np.mean((m_t - enzyme_activity(T_test).ravel()) ** 2))
print(f"coverage of the TRUE curve by the 95 % latent band : {cover_latent:.1%}  (target ≈ 95 %)")
print(f"coverage of fresh NOISY assays by the 95 % band     : {cover_pred:.1%}  (target ≈ 95 %)")
print(f"posterior-mean RMSE vs the true curve               : {rmse:.3f} activity units")

🔍 **What just happened?** With a reasonable kernel the coverage is honest: fresh noisy assays fall inside the 95 % band
about 95 % of the time (here a touch above, i.e. slightly *conservative*), and the smooth true curve is essentially always
inside — erring on the safe side, never over-confident. This is the headline feature a point estimate cannot give: a
prediction you can *bet on*. Two cautions we will keep testing: coverage is only this honest if the kernel and noise are
roughly right — a mis-specified kernel can make the band badly **over-confident** (Section 8) — and the "95 % band" for a
**measurement** must add the noise variance $\sigma^2$ to the latent variance, which is why `sd_pred` uses
$\sqrt{v+\sigma^2}$ (the latent band alone is for the underlying $f$, not the next noisy readout).

✏️ **Your turn.** Set the kernel `length` to 1.0 (very wiggly) and re-run 2.3–2.4. The mean overfits and the band becomes
*over-confident* — coverage drops well below 95 %. The length scale is not a nuisance; it controls calibration. Section 3
learns it from the data instead of guessing.

### 📄 Research connection — Gaussian processes and Bayesian optimisation

- **What the field does.** A Gaussian process is a prior over functions with a closed-form posterior (Rasmussen & Williams 2006 is the standard
text). Built on it, **Bayesian optimisation** — efficient global optimisation of expensive black-box functions — goes
back to Kushner (1964) and Močkus (1975), was popularised for engineering by Jones, Schonlau & Welch's EGO (1998), and
became a workhorse of machine learning for hyper-parameter tuning (Snoek, Larochelle & Adams 2012) with regret
guarantees for the UCB rule (Srinivas *et al.* 2010).
- **What this notebook builds.** The exact from-scratch GP of Section 2 and the acquisition functions of Section 4 are the real algorithms, just on tiny
data. Production libraries (GPyTorch, BoTorch, scikit-learn's `GaussianProcessRegressor`) add scalable linear algebra,
gradients through the acquisition, and many kernels — but the equations are the ones in the boxes above.
- **Where the frontier is.** As of 2025, active directions include GPs that scale to large $n$ (inducing-point and variational approximations),
deep-kernel and neural-network surrogates for structured inputs (sequences, graphs, molecules), high-dimensional and
batch/parallel BO, multi-fidelity BO that mixes cheap and expensive assays, and multi-objective BO — the methods behind
"self-driving labs."  A recurring open problem is trustworthy uncertainty *out of distribution*, where the surrogate is
most confidently wrong (Section 8, and Module 39).
- **Read next.** Rasmussen & Williams 2006, *Gaussian Processes for Machine Learning* (free online); Shahriari *et al.* 2016 (*Proc
IEEE*), *Taking the human out of the loop: a review of Bayesian optimization*; Jones, Schonlau & Welch 1998 (*J Glob
Optim*), *Efficient global optimization of expensive black-box functions*.

---
## 3 · Choosing the kernel: the marginal likelihood

> 📍 **Where we are.** We guessed the length scale, amplitude and noise. [Module 23](23_model_selection.ipynb) chose
> hyper-parameters by cross-validation. GPs offer a second, often cheaper route that uses *all* the data at once: the
> **marginal likelihood**, which has Occam's razor built in.

### 🧮 The log marginal likelihood

Integrating out the unknown function, the probability of the observed data under hyper-parameters
$\theta=(\ell,\,\text{signal},\,\sigma)$ is Gaussian with covariance $K_\theta+\sigma^2 I$:

$$\log p(y\mid X,\theta)= \underbrace{-\tfrac12\, y^\top (K_\theta+\sigma^2 I)^{-1} y}_{\text{data fit}}
\;\underbrace{-\;\tfrac12\log\lvert K_\theta+\sigma^2 I\rvert}_{\text{complexity penalty}}\;-\;\tfrac n2\log 2\pi.$$

*In words:* the first term rewards **fitting** the data (small when the model explains $y$); the second term **penalises
complexity** — a wiggly, short-length-scale model can fit anything but spreads its prior probability thinly over many
possible datasets, so $\log\lvert K+\sigma^2 I\rvert$ is large and it is punished. Maximising the sum automatically trades
fit against simplicity: **Occam's razor** falls out of the Gaussian, no validation set required. With the Cholesky factor
$L$ this is cheap: $y^\top(K+\sigma^2I)^{-1}y=\lVert L^{-1}y\rVert^2$ and $\log\lvert K+\sigma^2I\rvert=2\sum_i\log L_{ii}$.

In [ ]:
# ── 3.1 Log marginal likelihood, and fitting θ by maximising it ──
def log_marginal_likelihood(theta_log, X, y):
    """θ = exp(theta_log) = (length, signal, noise). Returns the log marginal likelihood of a zero-mean RBF GP."""
    length, signal, noise = np.exp(theta_log)
    yc = y - y.mean()
    K = rbf_kernel(X, X, length=length, signal=signal) + (noise**2 + 1e-8) * np.eye(len(X))
    try:
        L = cholesky(K, lower=True)
    except np.linalg.LinAlgError:
        return -1e25
    a = cho_solve((L, True), yc)
    return -0.5 * yc @ a - np.sum(np.log(np.diag(L))) - 0.5 * len(X) * np.log(2 * np.pi)


def fit_gp_hyperparameters(X, y, n_restarts=8, seed=SEED):
    """Maximise the log marginal likelihood from several random starts (it is non-convex)."""
    g = np.random.default_rng(seed)
    best = None
    for _ in range(n_restarts):
        x0 = g.normal([np.log(10), np.log(0.5), np.log(0.1)], 0.7)
        res = minimize(lambda t: -log_marginal_likelihood(t, X, y), x0, method="L-BFGS-B")
        if best is None or res.fun < best.fun:
            best = res
    return np.exp(best.x), -best.fun


(length_hat, signal_hat, noise_hat), lml = fit_gp_hyperparameters(T_obs, y_obs)
print(f"learned length ℓ = {length_hat:.2f} °C | signal = {signal_hat:.2f} | noise σ = {noise_hat:.3f}  "
      f"(true noise {TRUE_NOISE})")
print(f"log marginal likelihood at the optimum: {lml:.2f}")

In [ ]:
# ── 3.2 Occam's razor: the marginal likelihood as a function of length scale ──
lengths = np.linspace(1.5, 30, 60)
lml_curve = [log_marginal_likelihood(np.log([l, signal_hat, noise_hat]), T_obs, y_obs) for l in lengths]

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].plot(lengths, lml_curve, color="tab:purple")
axes[0].axvline(length_hat, ls="--", color="k", lw=1, label=f"maximum ℓ ≈ {length_hat:.1f}")
axes[0].set_xlabel("RBF length scale ℓ (°C)"); axes[0].set_ylabel("log marginal likelihood")
axes[0].set_title("Occam's razor: too wiggly (left) and too stiff (right) are both penalised")
axes[0].legend(fontsize=8)
for l, c, lab in [(2.0, "tab:red", "ℓ=2 (overfit)"), (length_hat, "tab:blue", f"ℓ={length_hat:.0f} (learned)"),
                  (30.0, "tab:green", "ℓ=30 (underfit)")]:
    m, v = gp_predict(T_obs, y_obs, T_grid, lambda A, B: rbf_kernel(A, B, length=l, signal=signal_hat), noise_hat)
    axes[1].plot(T_grid, m, color=c, lw=1.4, label=lab)
axes[1].plot(T_grid, enzyme_activity(T_grid), "k--", lw=1, label="truth")
axes[1].scatter(T_obs, y_obs, c="0.3", s=18, zorder=5)
axes[1].set_xlabel("temperature (°C)"); axes[1].set_ylabel("activity"); axes[1].legend(fontsize=7)
axes[1].set_title("The learned ℓ matches the truth best")
plt.tight_layout(); plt.show()

In [ ]:
# ── 3.3 Cross-check: length scale by 5-fold CV (Module 23) vs by marginal likelihood ──
def cv_rmse(length, X, y, k=5, seed=SEED):
    """k-fold cross-validated RMSE of the GP posterior mean for a given length scale."""
    g = np.random.default_rng(seed)
    idx = g.permutation(len(X))
    folds = np.array_split(idx, k)
    errs = []
    for f in folds:
        tr = np.setdiff1d(idx, f)
        m, _ = gp_predict(X[tr], y[tr], X[f], lambda A, B: rbf_kernel(A, B, length=length, signal=signal_hat), noise_hat)
        errs.append(np.mean((m - y[f]) ** 2))
    return np.sqrt(np.mean(errs))


cv_curve = [cv_rmse(l, T_obs, y_obs) for l in lengths]
cv_best = lengths[int(np.argmin(cv_curve))]
print(f"length scale chosen by marginal likelihood : {length_hat:5.1f} °C")
print(f"length scale chosen by 5-fold CV (min RMSE) : {cv_best:5.1f} °C")
print("Both routes land in the same neighbourhood — the marginal likelihood just needed no held-out split.")

🔍 **What just happened?** The marginal-likelihood curve has a clear peak: very small $\ell$ (an over-flexible model) and
very large $\ell$ (a too-rigid model) are *both* penalised, and the maximum sits near the width over which the true curve
actually varies (its features span roughly $\ell\approx9$ °C; the curve is a fixed function, not a GP draw, so this is the
scale the GP infers, not a "true" length scale). The right panel confirms it — the learned $\ell$ tracks the truth, while
$\ell=2$ overfits the noise and $\ell=30$
cannot bend enough. Reassuringly, cross-validation ([Module 23](23_model_selection.ipynb)) picks essentially the same
length scale, but the marginal likelihood got there without holding anything out, which matters when every data point cost
a day in the lab. The learned noise $\hat\sigma$ also recovers the true assay noise.

⚠️ **The marginal likelihood is not magic.** It assumes the *model class* (here, a stationary RBF) is right. If the true
function has a sharp transition (a denaturation cliff) the stationary RBF will compromise, and the marginal likelihood
will happily report a confident but wrong length scale. Treat it as a strong default, not a guarantee.

---
## 4 · Bayesian optimisation: which experiment next?

> 📍 **Where we are.** The GP gives a mean *and* a calibrated uncertainty everywhere. Bayesian optimisation turns that
> into a decision rule — an **acquisition function** — that scores every candidate experiment by how useful running it
> would be, then picks the best. We never need the true function's gradient; we only query it (expensively).

### 🧮 Three acquisition functions

Let $f^+$ be the best (largest) value observed so far and $\mu(x),\sigma(x)$ the GP posterior mean and standard deviation.
With $z=\dfrac{\mu(x)-f^+-\xi}{\sigma(x)}$ and $\Phi,\phi$ the standard-normal CDF and PDF:

$$\textbf{PI}(x)=\Phi(z),\qquad
\textbf{EI}(x)=\big(\mu(x)-f^+-\xi\big)\,\Phi(z)+\sigma(x)\,\phi(z),\qquad
\textbf{UCB}(x)=\mu(x)+\beta\,\sigma(x).$$

*In words:* **probability of improvement** asks only *whether* we beat the incumbent and so can be timid.
**Expected improvement** asks *by how much* — its two terms are exactly the exploit/explore split: the first rewards a high
mean, the second rewards high uncertainty, and EI is large either where the mean is promising **or** where we are ignorant.
**Upper confidence bound** is the simplest version of the same idea, with $\beta$ dialling exploration up or down (Srinivas
*et al.* 2010 give a schedule with regret guarantees). EI has a closed form because improvement is a truncated Gaussian;
let us derive it in code and watch it drive the search.

In [ ]:
# ── 4.1 Acquisition functions (for maximisation) ──
def expected_improvement(mu, sd, f_best, xi=0.01):
    """EI = (μ−f⁺−ξ)Φ(z) + σφ(z), with z=(μ−f⁺−ξ)/σ. Zero where σ=0 (nothing to learn)."""
    sd = np.maximum(sd, 1e-9)
    z = (mu - f_best - xi) / sd
    return (mu - f_best - xi) * norm.cdf(z) + sd * norm.pdf(z)


def probability_of_improvement(mu, sd, f_best, xi=0.01):
    """PI = Φ((μ−f⁺−ξ)/σ): the probability of beating the incumbent."""
    return norm.cdf((mu - f_best - xi) / np.maximum(sd, 1e-9))


def upper_confidence_bound(mu, sd, beta=2.0):
    """UCB = μ + β·σ: optimistic in the face of uncertainty."""
    return mu + beta * sd


print("EI, PI and UCB defined. All three take the GP's μ and σ at candidate points and return a score to maximise.")

In [ ]:
# ── 4.2 One Bayesian-optimisation step, drawn ──
rng = np.random.default_rng(SEED + 4)
T_bo = rng.uniform(15, 68, 4)[:, None]                          # 4 experiments done so far
y_bo = enzyme_activity(T_bo).ravel() + rng.normal(0, TRUE_NOISE, 4)
th, _ = fit_gp_hyperparameters(T_bo, y_bo, seed=SEED + 4)
kern_bo = lambda A, B: rbf_kernel(A, B, length=th[0], signal=th[1])
mu, var = gp_predict(T_bo, y_bo, T_grid, kern_bo, noise=th[2]); sd = np.sqrt(var)
ei = expected_improvement(mu, sd, y_bo.max())
x_next = float(T_grid[np.argmax(ei), 0])

fig, (a1, a2) = plt.subplots(2, 1, figsize=(6.6, 4.6), sharex=True, height_ratios=[2, 1])
a1.plot(T_grid, enzyme_activity(T_grid), "k--", lw=1, label="truth (hidden)")
a1.plot(T_grid, mu, "tab:blue", label="GP mean")
a1.fill_between(T_grid.ravel(), mu - 1.96 * sd, mu + 1.96 * sd, color="tab:blue", alpha=0.2)
a1.scatter(T_bo, y_bo, c="tab:red", zorder=5, label="done"); a1.legend(fontsize=7); a1.set_ylabel("activity")
a1.set_title("Bayesian optimisation: EI proposes the next temperature to assay")
a2.plot(T_grid, ei, "tab:green"); a2.axvline(x_next, ls="--", color="k", lw=1)
a2.fill_between(T_grid.ravel(), 0, ei.ravel(), color="tab:green", alpha=0.2)
a2.set_ylabel("EI"); a2.set_xlabel("temperature (°C)")
plt.tight_layout(); plt.show()
print("EI's pick for the next assay:", round(x_next, 1), "°C "
      "(balancing the region the mean already finds promising against where the band is still wide).")

In [ ]:
# ── 4.3 A 2-D SIMULATED bioprocess: yield vs temperature and pH, hidden optimum ──
def culture_yield(temp, pH):
    """True (noise-free) relative yield of a culture vs temperature (°C) and pH. One broad hidden optimum + a ridge."""
    temp, pH = np.asarray(temp, float), np.asarray(pH, float)
    peak = np.exp(-0.5 * (((temp - 33.0) / 4.5) ** 2 + ((pH - 6.9) / 0.45) ** 2))
    ridge = 0.25 * np.exp(-0.5 * (((temp - 28.0) / 9.0) ** 2 + ((pH - 7.6) / 0.9) ** 2))
    return peak + ridge


T_RANGE, PH_RANGE = (24.0, 42.0), (5.5, 8.0)
gt, gp_ = np.meshgrid(np.linspace(*T_RANGE, 45), np.linspace(*PH_RANGE, 45))
CAND = np.c_[gt.ravel(), gp_.ravel()]                           # 2,025 candidate conditions (a plate-sized menu)
Z = culture_yield(CAND[:, 0], CAND[:, 1])
opt_idx = int(np.argmax(Z)); X_OPT, Y_OPT = CAND[opt_idx], Z[opt_idx]
print(f"TRUE optimum on the candidate grid: {X_OPT[0]:.1f} °C, pH {X_OPT[1]:.2f}  →  yield {Y_OPT:.3f}")

In [ ]:
# ── 4.4 Compare strategies: EI, UCB, random and grid, averaged over seeds ──
N_INIT, N_ITER = 4, 22                                          # 4 initial + 22 proposed = 26 experiments total
# A genuine coarse 6×5 grid over the (temperature, pH) box, visited in a raster sweep (model-free baseline).
_gtc, _gpc = np.meshgrid(np.linspace(*T_RANGE, 6), np.linspace(*PH_RANGE, 5))
GRID_ORDER = [int(np.argmin(((CAND - p) ** 2).sum(1))) for p in np.c_[_gtc.ravel(), _gpc.ravel()]]

def run_bo(strategy, n_init=N_INIT, n_iter=N_ITER, seed=0):
    """Return the best TRUE yield found after each experiment, for one strategy on the 2-D culture problem."""
    g = np.random.default_rng(seed)
    scaler = lambda A: (A - [T_RANGE[0], PH_RANGE[0]]) / [T_RANGE[1] - T_RANGE[0], PH_RANGE[1] - PH_RANGE[0]]
    chosen = GRID_ORDER[:n_init] if strategy == "grid" else list(g.choice(len(CAND), n_init, replace=False))
    yobs = culture_yield(CAND[chosen, 0], CAND[chosen, 1]) + g.normal(0, 0.03, len(chosen))
    best = [Z[chosen].max()]
    for t in range(n_iter):
        if strategy == "random":
            nxt = g.choice(np.setdiff1d(np.arange(len(CAND)), chosen))
        elif strategy == "grid":
            nxt = GRID_ORDER[len(chosen) % len(GRID_ORDER)]     # next coarse-grid point in the sweep
        else:
            th = [0.25, float(np.std(yobs)) + 1e-3, 0.03]       # fixed sensible kernel: cheap & stable in the loop
            kf = lambda A, B: rbf_kernel(scaler(A), scaler(B), length=th[0], signal=th[1])
            mu, var = gp_predict(CAND[chosen], np.asarray(yobs), CAND, kf, noise=th[2]); sd = np.sqrt(var)
            acq = expected_improvement(mu, sd, np.max(yobs)) if strategy == "EI" else upper_confidence_bound(mu, sd, 2.0)
            acq[chosen] = -np.inf                                # do not repeat an experiment
            nxt = int(np.argmax(acq))
        chosen.append(int(nxt))
        yobs = np.append(yobs, culture_yield(CAND[nxt, 0], CAND[nxt, 1]) + g.normal(0, 0.03))
        best.append(Z[chosen].max())                            # best TRUE (noise-free) yield secured so far
    return np.array(best)


SEEDS = range(10)
curves = {s: np.mean([run_bo(s, seed=k) for k in SEEDS], axis=0) for s in ["EI", "UCB", "random", "grid"]}
thr = 0.99 * Y_OPT                                              # "near-optimal" = within 1 % of the true optimum
print(f"Averaged over {len(list(SEEDS))} seeds (true optimum {Y_OPT:.3f}; each curve indexed by total experiments):")
print(f"  {'strategy':8s} {'after 6 exp':>12s} {'after 26 exp':>13s} {'experiments to reach 99% of optimum':>36s}")
for s, c in curves.items():
    hit = int(np.argmax(c >= thr)) if (c >= thr).any() else None
    print(f"  {s:8s} {c[2]:12.3f} {c[-1]:13.3f} {(str(N_INIT + hit) if hit is not None else 'not within 26'):>36s}")

In [ ]:
# ── 4.5 Learning curves and a sampled EI trajectory ──
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.6))
for s, c in curves.items():
    a1.plot(N_INIT + np.arange(len(c)), c, marker="o", ms=3, label=s)
a1.axhline(Y_OPT, ls="--", color="k", lw=1, label="true optimum")
a1.set_xlabel("number of experiments"); a1.set_ylabel("best true yield found")
a1.set_title("Bayesian optimisation reaches the optimum in fewer experiments"); a1.legend(fontsize=8)

g = np.random.default_rng(SEED + 7)
init = g.choice(len(CAND), 4, replace=False); chosen = list(init)
yobs = culture_yield(CAND[chosen, 0], CAND[chosen, 1]) + g.normal(0, 0.03, 4)
scaler = lambda A: (A - [T_RANGE[0], PH_RANGE[0]]) / [T_RANGE[1] - T_RANGE[0], PH_RANGE[1] - PH_RANGE[0]]
for t in range(18):
    kf = lambda A, B: rbf_kernel(scaler(A), scaler(B), length=0.25, signal=float(np.std(yobs)) + 1e-3)
    mu, var = gp_predict(CAND[chosen], np.asarray(yobs), CAND, kf, noise=0.03)
    acq = expected_improvement(mu, np.sqrt(var), np.max(yobs)); acq[chosen] = -np.inf
    nxt = int(np.argmax(acq)); chosen.append(nxt)
    yobs = np.append(yobs, culture_yield(CAND[nxt, 0], CAND[nxt, 1]) + g.normal(0, 0.03))
a2.contourf(gt, gp_, Z.reshape(gt.shape), levels=15, cmap="viridis", alpha=0.9)
a2.scatter(CAND[chosen, 0], CAND[chosen, 1], c=range(len(chosen)), cmap="autumn", s=22, edgecolor="k", linewidth=0.3)
a2.scatter(*X_OPT, marker="*", s=240, c="white", edgecolor="k", label="true optimum")
a2.set_xlabel("temperature (°C)"); a2.set_ylabel("pH"); a2.set_title("EI trajectory (dark→bright = early→late)")
a2.legend(fontsize=8)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Averaged over ten seeds, **EI and UCB reach within 1 % of the true optimum in about 11
experiments** and essentially hit it by 26 (best ≈ 1.159 and 1.160, optimum 1.160). Neither baseline gets there: random
search climbs to only ≈ 1.05 after 26 experiments and the coarse space-filling grid to ≈ 1.04 — **neither comes within
1 % of the optimum in the budget**. Instead of spreading effort evenly, BO pours it into the promising region once the GP
has located it, while still probing the uncertain ridge. The trajectory plot shows the tell-tale behaviour — a few
scattered early points (exploration), then a tight cluster homing in on the hidden optimum (exploitation). This is the loop
that machine-learning-guided experimentation runs in the wet lab, with each dot a real assay.

✏️ **Your turn.** Change UCB's `beta` from 2.0 to 0.5 (greedy) and 5.0 (very exploratory) in `run_bo`. Greedy UCB can
stall on the smaller ridge; over-exploratory UCB wastes experiments far from any optimum. EI needs no such knob, which is
one reason it is the default.

---
## 5 · Batches of experiments: filling a 96-well plate

> 📍 **Where we are.** Real labs do not run one experiment, wait, then run the next — they fill a **96-well plate** and
> read it all at once. Picking the single best point 96 times gives 96 copies of the same well. We need a batch of 96
> *different, jointly useful* experiments from one GP.

Two cheap, popular tricks turn a sequential acquisition into a batch:

* **Thompson sampling.** Draw a *sample function* from the GP posterior and pick its maximiser; repeat 96 times. Each draw
  is a different plausible world, so different draws favour different wells — exploration comes for free from posterior
  randomness (Russo *et al.* 2018).
* **Constant liar.** Pick the best point by EI, then *pretend* we already measured it, assigning it one **fixed constant**
  value — the "lie" — reused for every pick in the batch (here the mean of the real measurements). Re-fit and pick again:
  the fantasised observation suppresses EI near the chosen well, so the next pick moves elsewhere (Ginsbourger *et al.*
  2010). Using a *constant* (rather than the GP's own posterior mean at each pick, which is the related "Kriging believer"
  variant) is what makes this the constant liar.

In [ ]:
# ── 5.1 Select a batch of 96 wells by Thompson sampling and by the constant liar ──
scaler = lambda A: (A - [T_RANGE[0], PH_RANGE[0]]) / [T_RANGE[1] - T_RANGE[0], PH_RANGE[1] - PH_RANGE[0]]
rng = np.random.default_rng(SEED + 8)
init = rng.choice(len(CAND), 8, replace=False)
yobs = culture_yield(CAND[init, 0], CAND[init, 1]) + rng.normal(0, 0.03, 8)
kf = lambda A, B: rbf_kernel(scaler(A), scaler(B), length=0.25, signal=float(np.std(yobs)) + 1e-3)

def thompson_batch(chosen, yobs, size=96):
    """Draw `size` posterior sample functions over the candidate menu; each well = one sample's maximiser."""
    mu, _, cov = gp_predict(CAND[chosen], yobs, CAND, kf, noise=0.03, return_cov=True)
    Lc = cholesky(cov + 1e-6 * np.eye(len(CAND)), lower=True)
    picks = []
    for _ in range(size):
        f = mu + Lc @ np.random.default_rng(len(picks) + SEED).standard_normal(len(CAND))
        f[picks] = -np.inf
        picks.append(int(np.argmax(f)))
    return picks

def liar_batch(chosen, yobs, size=96):
    """Greedy EI with the 'constant liar' (Ginsbourger et al. 2010): after each pick, insert ONE fixed constant
    (the mean of the real measurements) in its place, re-fit, and pick again, so the next EI avoids it."""
    ch, yy = list(chosen), list(yobs); picks = []
    lie = float(np.mean(yobs))                                   # a single constant reused for every fantasised pick
    for _ in range(size):
        mu, var = gp_predict(CAND[ch], np.asarray(yy), CAND, kf, noise=0.03)
        acq = expected_improvement(mu, np.sqrt(var), np.max(yy)); acq[ch] = -np.inf
        nxt = int(np.argmax(acq)); picks.append(nxt)
        ch.append(nxt); yy.append(lie)                          # the constant "lie"
    return picks

ts = thompson_batch(list(init), yobs); cl = liar_batch(list(init), yobs)
print(f"Thompson batch: {len(set(ts))}/96 distinct wells, best true yield in batch {Z[ts].max():.3f}")
print(f"Constant-liar batch: {len(set(cl))}/96 distinct wells, best true yield in batch {Z[cl].max():.3f}")

In [ ]:
# ── 5.2 One plate (96 parallel) vs 96 sequential experiments: cost of parallelism ──
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
for ax, picks, name in [(axes[0], ts, "Thompson batch (1 plate of 96)"), (axes[1], cl, "constant-liar batch (1 plate)")]:
    ax.contourf(gt, gp_, Z.reshape(gt.shape), levels=15, cmap="viridis", alpha=0.85)
    ax.scatter(CAND[picks, 0], CAND[picks, 1], c="white", s=12, edgecolor="k", linewidth=0.3)
    ax.scatter(*X_OPT, marker="*", s=200, c="red", edgecolor="k")
    ax.set_title(name); ax.set_xlabel("temperature (°C)"); ax.set_ylabel("pH")
plt.tight_layout(); plt.show()

seq = np.mean([run_bo("EI", n_init=8, n_iter=88, seed=k)[-1] for k in range(2)])
print(f"96 SEQUENTIAL EI experiments reach best true yield ≈ {seq:.3f} (optimum {Y_OPT:.3f}).")
print(f"One parallel plate gets close in a SINGLE round: Thompson {Z[ts].max():.3f}, constant-liar {Z[cl].max():.3f}.")
print("The batch trades a little efficiency per experiment for a huge saving in rounds; that trade is usually worth it.")

🔍 **What just happened?** Both batch rules spread 96 wells across the surface instead of stacking them on one spot:
Thompson sampling scatters them according to posterior plausibility, the constant liar lays down a greedy but diverse set.
On this surface the Thompson plate brackets the optimum (best in-plate ≈ 1.16) and the constant-liar plate comes close
(≈ 1.08) — a single parallel plate recovers most of what 96 patient *sequential* experiments reach, but in **one round
instead of ninety-six**, and wall-clock rounds (not total wells) are usually what a project cannot afford. That gap between
batch and fully sequential BO is the price of parallelism; shrinking it is an active research topic (Section 9).

✏️ **Your turn.** Reduce the plate to 24 wells and run three rounds (re-fit the GP on the real readouts after each plate).
Three rounds of 24 usually beat one plate of 96 for the same total count — rounds buy information. Where is the sweet spot
for your real constraints?

---
## 6 · Protein engineering over a combinatorial library (MLDE)

> 📍 **Where we are.** We optimised a 2-D continuous surface. Protein engineering optimises over **sequences**: a
> machine-learning-guided directed-evolution (MLDE) loop measures a batch of variants, fits a surrogate, proposes the next
> batch, and repeats. We reproduce the classic **GB1** four-site system: all $20^4 = 160{,}000$ combinations at four
> positions of protein G's B1 domain, a landscape riddled with **epistasis** (mutations whose effects depend on each other)
> that was exhaustively **measured** by Wu *et al.* (2016) and later used to benchmark **ML-assisted directed evolution** by
> Wu *et al.* (2019).

🔬 **Assay box — a four-site combinatorial library.** Pick four positions in a protein and make every combination of the
20 amino acids at those sites (160,000 variants), leaving the rest as wild type. Each variant's "fitness" (binding,
stability, activity) is measured in a pooled assay. Because the four sites touch in 3-D, the landscape carries
**epistasis**: the best residue at one site depends partly on the others. We build it from the toolkit's Potts model (the
same model Module 37 fits to real families); the **true fields $h$ and couplings $J$ are the oracle**, and we deliberately
choose the four positions with the strongest mutual couplings. Most of the fitness here still turns out to be *additive*
(a sum of per-site effects), with a smaller epistatic remainder — which, as we will see, decides whether a flexible
surrogate is worth its extra variance.

In [ ]:
# ── 6.1 Build the SIMULATED four-site landscape (true oracle = the Potts model) ──
rng = np.random.default_rng(SEED)
structure = simulate_helix_bundle()
contacts = structure["contacts"]
family = simulate_protein_family(rng, contacts, n_seqs=2000, n_sweeps=80)    # ≈ 4 s
h_true, J_true, wild_type = family["h"], family["J"], family["wild_type"]
L = len(wild_type)

# choose 4 positions with the strongest mutual couplings (so their effects interact = epistasis)
from itertools import combinations
Jmag = np.abs(J_true).sum(axis=(2, 3))                          # total |coupling| between each pair of positions
Jmag[~np.triu(np.ones((L, L), bool), 1)] = 0
cands = np.argsort(-Jmag.sum(0))[:14]
sites = np.array(max(combinations(sorted(int(c) for c in cands), 4),
                     key=lambda c: Jmag[np.ix_(list(c), list(c))].sum()))
n_pairs = int(np.triu(contacts[np.ix_(sites, sites)], 1).sum())
print(f"Protein length L = {L}; engineering 4 sites {sites} with {n_pairs} contacting (coupled) pairs among them.")

GRID = np.array(np.meshgrid(*[np.arange(20)] * 4, indexing="ij")).reshape(4, -1).T   # 160,000 × 4
LIB = np.tile(wild_type, (len(GRID), 1)); LIB[:, sites] = GRID
TRUE_FIT = potts_score(LIB, h_true, J_true)                     # the ground-truth fitness of all 160,000 variants
FIT_NOISE = 0.5                                                 # assay noise (std), on the Potts energy scale
g_opt = int(np.argmax(TRUE_FIT))
print(f"Global optimum: variant {''.join(AMINO_ACIDS[a] for a in GRID[g_opt])} with true fitness {TRUE_FIT[g_opt]:.2f}")
print(f"Fitness spread over the library: {TRUE_FIT.min():.1f} … {TRUE_FIT.max():.1f}")

# How much of the landscape is *additive*? Fit the best-possible additive (per-site) model to ALL 160k variants.
OH_all = np.zeros((len(GRID), 80), np.float32)
OH_all[np.arange(len(GRID))[:, None], GRID + np.arange(4) * 20] = 1.0
beta, *_ = np.linalg.lstsq(np.c_[np.ones(len(GRID)), OH_all], TRUE_FIT, rcond=None)
add_fit = np.c_[np.ones(len(GRID)), OH_all] @ beta
add_ceiling = 1 - np.var(TRUE_FIT - add_fit) / np.var(TRUE_FIT)
print(f"Best possible ADDITIVE model explains R² = {add_ceiling:.3f} of the landscape → "
      f"~{1 - add_ceiling:.0%} of the fitness variance is epistasis no additive model can ever reach.")

In [ ]:
# ── 6.2 Features and two GP kernels: additive (linear) vs epistatic (RBF on one-hot) ──
OH = OH_all                                                     # one-hot of the 4 variable sites (4 × 20), from 6.1

def matches(idx_a, idx_b):
    """#positions (out of 4) at which two sets of variants share the same amino acid = OH·OHᵀ."""
    return OH[idx_a] @ OH[idx_b].T

def gp_on_library(obs_idx, y_obs, kind, cand_idx, want_sd=True):
    """Posterior mean (and sd) over the candidate variants `cand_idx`, with a Hamming-based kernel.
    kind='linear' → additive (base k = #matches + 1); kind='rbf' → epistatic (base k = exp(−(4−#matches)/ℓ)).
    Kept in float32 so even the full 160k pool fits comfortably in memory."""
    obs_idx, cand_idx = np.asarray(obs_idx), np.asarray(cand_idx)
    B_oo = matches(obs_idx, obs_idx)                           # n × n (float32)
    B_ao = matches(cand_idx, obs_idx)                         # |cand| × n (float32)
    if kind == "linear":
        B_oo = B_oo + 1.0; np.add(B_ao, 1.0, out=B_ao); kdiag = 5.0
    else:
        ell = 1.5
        B_oo = np.exp(-(4 - B_oo) / ell); np.exp(-(4 - B_ao) / ell, out=B_ao); kdiag = 1.0
    sig2 = float(y_obs.var()) + 1e-6
    K = sig2 * B_oo + (FIT_NOISE**2 + 1e-6) * np.eye(len(obs_idx))
    Minv = np.linalg.inv(K).astype(np.float32)
    alpha = (Minv @ (y_obs - y_obs.mean())).astype(np.float32)
    mean = sig2 * (B_ao @ alpha) + y_obs.mean()
    if not want_sd:
        return mean.astype(np.float64), None
    red = np.einsum("ij,ij->i", B_ao @ Minv, B_ao)           # diag(B_ao Minv B_aoᵀ)
    var = np.maximum(sig2 * kdiag - sig2**2 * red, 1e-9)
    return mean.astype(np.float64), np.sqrt(var).astype(np.float64)

print("A linear kernel sums per-site similarity (additive); the RBF of #matches is nonlinear, so it can feel epistasis.")

In [ ]:
# ── 6.3 🧪 Ground-truth check: MLDE rounds of 96 — epistatic vs additive vs random ──
def mlde(strategy, n_round=96, rounds=5, pool=25000, seed=0):
    """One MLDE run. Returns best TRUE fitness found after each round and the final design rank of the global optimum.
    Each round screens a fresh random candidate pool (as a real campaign screens a subset, not all 160k)."""
    g = np.random.default_rng(seed)
    kind = "rbf" if strategy == "epistatic" else "linear"
    measured = list(g.choice(len(GRID), n_round, replace=False))         # round 0: random batch
    y = TRUE_FIT[measured] + g.normal(0, FIT_NOISE, len(measured))
    best = [TRUE_FIT[measured].max()]
    for _ in range(rounds - 1):
        cand = np.setdiff1d(g.choice(len(GRID), pool, replace=False), measured)   # a round's candidate menu
        if strategy == "random":
            nxt = list(g.choice(cand, n_round, replace=False))
        else:
            mean, sd = gp_on_library(np.array(measured), np.asarray(y), kind, cand)
            nxt = list(cand[np.argsort(upper_confidence_bound(mean, sd, 2.0))[::-1][:n_round]])
        measured += nxt
        y = np.append(y, TRUE_FIT[nxt] + g.normal(0, FIT_NOISE, len(nxt)))
        best.append(TRUE_FIT[measured].max())
    rank = len(GRID)
    if strategy != "random":                                              # rank the true optimum in the final surrogate
        mean, _ = gp_on_library(np.array(measured), np.asarray(y), kind, np.arange(len(GRID)), want_sd=False)
        rank = int((mean > mean[g_opt]).sum()) + 1
    return np.array(best), rank, g_opt in measured

t0 = time.time()
res = {s: [mlde(s, seed=k) for k in range(5)] for s in ["epistatic", "additive", "random"]}
summ = {s: (np.mean([r[0] for r in runs], axis=0), np.median([r[1] for r in runs]),
            np.mean([r[2] for r in runs])) for s, runs in res.items()}
print(f"Averaged over 5 seeds, 5 rounds of 96 (480 of 160,000 variants assayed = {480/160000:.1%}).  ({time.time()-t0:.0f} s)")
print(f"global optimum true fitness = {TRUE_FIT[g_opt]:.2f}\n")
for s, (curve, rank, hit) in summ.items():
    print(f"  {s:9s}: best true fitness found {curve[-1]:.2f} | optimum found outright {hit:.0%} of runs | "
          f"median rank of the true optimum in surrogate {int(rank) if rank==rank else 'NA'}")

In [ ]:
# ── 6.4 Learning curves: fitness secured vs number of variants assayed ──
fig, ax = plt.subplots(figsize=(6.4, 3.8))
xs = np.arange(5) * 96 + 96
for s, (curve, _, _) in summ.items():
    ax.plot(xs, curve, marker="o", ms=4, label=s)
ax.axhline(TRUE_FIT[g_opt], ls="--", color="k", lw=1, label="global optimum")
ax.set_xlabel("variants assayed"); ax.set_ylabel("best true fitness found")
ax.set_title("MLDE: a GP surrogate + UCB beats random mutagenesis\n(SIMULATED 4-site Potts landscape)")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

In [ ]:
# ── 6.5 🧪 Ground-truth check: surrogate accuracy, and the additive ceiling ──
from scipy.stats import spearmanr
g = np.random.default_rng(SEED + 12)
train = g.choice(len(GRID), 400, replace=False)                 # one realistic labelled set (400 variants)
y_train = TRUE_FIT[train] + g.normal(0, FIT_NOISE, 400)
held = g.choice(np.setdiff1d(np.arange(len(GRID)), train), 20000, replace=False)
for kind, label in [("linear", "additive (linear kernel)"), ("rbf", "epistatic (RBF kernel)")]:
    m, _ = gp_on_library(train, y_train, kind, held, want_sd=False)
    print(f"  {label:26s}: held-out Spearman {spearmanr(m, TRUE_FIT[held]).statistic:.3f}  "
          f"R² {1 - np.var(TRUE_FIT[held] - m) / np.var(TRUE_FIT[held]):.3f}")
print(f"  best-possible additive fit : R² {add_ceiling:.3f}  (the ceiling any additive model is stuck under)")

🔍 **What just happened? An honest result, and a subtler lesson.** Both GP surrogates, driven by UCB, find a top variant
from only ~0.3 % of the 160,000-variant library and crush random mutagenesis — that is the real promise of
machine-learning-guided directed evolution (Romero *et al.* 2013; Wu *et al.* 2019). But the two kernels perform *about the
same* on this landscape, and the accuracy check (6.5) shows why: most of the fitness here is **additive**, so the simple
additive (linear-kernel) surrogate is already well specified, and the flexible RBF's extra freedom buys little — sometimes
it even loses, paying in variance (overfitting the noise) for epistasis that is only a small slice of the signal. This is
the bias–variance trade-off of [Module 23](23_model_selection.ipynb), in sequence space: **more flexibility is not
automatically better** — the right inductive bias is the one that matches the biology.

The epistasis still matters, and the ceiling makes it concrete: no additive model can exceed the **ceiling R²** on the last
printed line, because a fixed fraction of the fitness lives in residue *combinations* it cannot represent (our finite-data
additive surrogate sits just under it). On a strongly epistatic landscape
(real GB1 is far more epistatic than our toy), that ceiling bites hard and an interaction-aware surrogate — RBF/Hamming
kernels, pairwise features, or protein-language-model embeddings ([Module 37](37_protein_sequences_language_models.ipynb)) —
pulls clearly ahead.

⚠️ **The same warning as always.** The surrogate is trusted to rank *unmeasured* variants. When UCB pushes designs into
regions the model has never seen, it can be confidently wrong — and optimisation will happily exploit that error. We
provoke and fix this failure mode head-on in [Module 39 · Generative design](39_generative_design.ipynb).

✏️ **Your turn.** In 6.3, replace UCB with a **greedy** rule (`mean` only, no `sd` term) and re-run. Greedy exploits the
surrogate and can stall below UCB's best-found, because it never probes uncertain variants — the exploration term earns its
keep here too.

### 📄 Research connection — Machine-learning-guided protein engineering

- **What the field does.** Directed evolution (Arnold, Nobel Prize 2018) improves proteins by iterated mutation and selection. **Machine-learning-
guided** directed evolution adds a surrogate that predicts fitness and proposes variants: Romero, Krause & Arnold (2013)
used Gaussian processes on a P450 landscape; the exhaustively measured GB1 four-site landscape (Wu *et al.* 2016) let
Wu *et al.* (2019) show that MLDE over four-site libraries finds high-fitness variants from small samples; Yang, Wu &
Arnold (2019) reviewed the loop; Hie & Yang (2022) survey adaptive experimentation for proteins.
- **What this notebook builds.** Section 6 is MLDE in miniature: a GP surrogate with a sequence kernel, UCB acquisition, and rounds of 96, scored
against a Potts-model oracle whose epistasis is known. Production MLDE often uses richer features — one-hot, physico-
chemical descriptors, or embeddings from a protein language model (Module 37) — feeding ridge, random forests, ensembles,
or GPs.
- **Where the frontier is.** As of 2025, open questions include which representation to trust (zero-shot language-model scores vs supervised
surrogates), how to budget exploration under strong epistasis, batch and multi-round design under tight assay budgets,
and combining sequence BO with structure-based and generative design (Module 39). Public landscapes such as ProteinGym
and FLIP let methods be benchmarked honestly.
- **Read next.** Romero, Krause & Arnold 2013 (*PNAS*), *Navigating the protein fitness landscape with Gaussian processes*; Wu *et al.*
2019 (*PNAS*), *Machine learning-assisted directed protein evolution with combinatorial libraries*; Yang, Wu & Arnold
2019 (*Nat Methods*), *Machine-learning-guided directed evolution for protein engineering*.

---
## 7 · Active learning: spend a labelling budget wisely

> 📍 **Where we are.** Bayesian optimisation chooses experiments to *maximise* a quantity. **Active learning** chooses
> experiments to *learn a model* fastest — same idea (query where you are most uncertain), different goal. The classic
> biological case: a human expert can annotate only so many cells; which ones teach the classifier most?

In [ ]:
# ── 7.1 A SIMULATED annotation task: PBMC cell types from single-cell profiles ──
rng = np.random.default_rng(SEED + 20)
sc = simulate_scrna(rng, n_cells=2000, n_genes=300, n_donors=4, n_batches=1, depth=350)  # shallow → classes overlap
keep = sc["obs"].true_type.to_numpy(str)                        # ground-truth labels (an expert would provide these)
X_all = pca(standardize(log_normalize(sc["counts"]))[0], n_components=8)[0]    # 8-D PCA features
classes = list(np.unique(keep))
y_all = np.array([classes.index(t) for t in keep])
tr_idx, te_idx = np.arange(1300), np.arange(1300, 2000)
print(f"{len(classes)} cell types: {classes}")
print(f"pool to label: {len(tr_idx)} cells | held-out test: {len(te_idx)} cells | features: 8 PCs")

In [ ]:
# ── 7.2 Uncertainty sampling vs random labelling ──
def active_learn(strategy, n_seed=12, n_query=12, rounds=14, seed=0):
    """Pool-based active learning. Returns (labels used, test accuracy) after each round."""
    g = np.random.default_rng(seed)
    labelled = list(g.choice(tr_idx, n_seed, replace=False))
    accs, budgets = [], []
    for _ in range(rounds):
        clf = LogisticRegression(max_iter=500, C=1.0).fit(X_all[labelled], y_all[labelled])
        pred = clf.predict(X_all[te_idx])
        accs.append(np.mean(pred == y_all[te_idx])); budgets.append(len(labelled))
        pool = np.setdiff1d(tr_idx, labelled)
        if strategy == "random":
            pick = g.choice(pool, min(n_query, len(pool)), replace=False)
        else:                                                   # least-confidence: smallest top-class probability
            p = clf.predict_proba(X_all[pool])
            pick = pool[np.argsort(p.max(1))[:n_query]]
        labelled += list(pick)
    return np.array(budgets), np.array(accs)

curves_al = {s: [active_learn(s, seed=k) for k in range(5)] for s in ["uncertainty", "random"]}
for s, runs in curves_al.items():
    acc = np.mean([r[1] for r in runs], axis=0); b = runs[0][0]
    mid = len(b) // 2
    print(f"{s:11s}: accuracy at {b[0]} labels {acc[0]:.3f} → at {b[mid]} {acc[mid]:.3f} → at {b[-1]} {acc[-1]:.3f}")

In [ ]:
# ── 7.3 Learning curves, and a case where uncertainty sampling wastes the budget ──
# Build a pool polluted with low-quality OUTLIERS (damaged, high-mitochondrial cells) — a real QC problem.
rng = np.random.default_rng(SEED + 21)
sc2 = simulate_scrna(rng, n_cells=2200, n_genes=300, n_donors=4, n_batches=1, depth=350, damaged_rate=0.12)
keep2 = sc2["obs"].true_type.to_numpy(str); damaged = sc2["obs"].is_damaged.to_numpy()
X2 = pca(standardize(log_normalize(sc2["counts"]))[0], n_components=8)[0]
y2 = np.array([classes.index(t) for t in keep2])
tr2, te2 = np.arange(1500), np.arange(1500, 2200)
te_healthy = te2[~damaged[te2]]                                 # we care about labelling *real* cells correctly

def active_learn_outliers(strategy, seed=0, n_seed=14, n_query=12, rounds=14):
    """Returns accuracy on healthy test cells and the fraction of each query batch wasted on damaged outliers."""
    g = np.random.default_rng(seed); lab = list(g.choice(tr2, n_seed, replace=False)); acc, wasted = [], []
    for _ in range(rounds):
        clf = LogisticRegression(max_iter=500).fit(X2[lab], y2[lab])
        acc.append(np.mean(clf.predict(X2[te_healthy]) == y2[te_healthy]))
        pool = np.setdiff1d(tr2, lab)
        if strategy == "random":
            pick = g.choice(pool, n_query, replace=False)
        else:
            p = clf.predict_proba(X2[pool]); pick = pool[np.argsort(p.max(1))[:n_query]]
        wasted.append(np.mean(damaged[pick])); lab += list(pick)
    return np.array(acc), np.array(wasted)

out_al = {s: [active_learn_outliers(s, k) for k in range(5)] for s in ["uncertainty", "random"]}

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.6))
for s, runs in curves_al.items():
    b = runs[0][0]; acc = np.array([r[1] for r in runs])
    a1.plot(b, acc.mean(0), marker="o", ms=3, label=s)
    a1.fill_between(b, acc.mean(0) - acc.std(0), acc.mean(0) + acc.std(0), alpha=0.15)
a1.set_xlabel("cells labelled"); a1.set_ylabel("test accuracy")
a1.set_title("Clean pool: uncertainty sampling learns a little faster"); a1.legend(fontsize=8)
bb = np.arange(14) * 12 + 14
for s, runs in out_al.items():
    a2.plot(bb, np.mean([r[1] for r in runs], axis=0), marker="s", ms=3, label=s)
a2.set_xlabel("cells labelled"); a2.set_ylabel("fraction of queries that are damaged outliers")
a2.set_title("Polluted pool: uncertainty sampling chases the junk"); a2.legend(fontsize=8)
plt.tight_layout(); plt.show()
wasted_u = np.mean([r[1] for r in out_al["uncertainty"]], axis=0)
print(f"With 12 % damaged cells in the pool, uncertainty sampling spends up to {wasted_u.max():.0%} of a query batch on "
      f"damaged outliers (random spends ~12 %, the base rate).")

🔍 **What just happened?** On the clean pool (left), **uncertainty sampling** — label the cells whose top-class probability
is lowest, i.e. those nearest the decision boundaries — reaches a given accuracy with a few fewer labels than random
annotation. The gain is real but modest here, because clean PBMC types separate easily (marker genes do most of the work);
active learning pays off most when labels are expensive *and* the classes genuinely overlap. This is why single-cell
label-transfer tools offer it but do not rely on it blindly.

The right panel shows the classic failure. When the pool is polluted with **outliers** — here damaged, high-mitochondrial
cells that belong to no clean type — least-confidence sampling pours its budget into them: those junk cells sit far from
every class and so look *maximally uncertain*, exactly the trap. A large fraction of each early query batch is wasted on
damaged cells (far above their 12 % base rate), while random sampling wastes only the base rate. Our strong marker genes
keep accuracy from collapsing, but the budget — the whole point of active learning — is squandered. Pure uncertainty
chases *ambiguity*, which is not the same as *informativeness*. Fixes: filter QC-fail cells first, down-weight by local
density (outliers live in sparse regions), or add a diversity/representativeness term — the active-learning cousin of the
batch-diversity problem from Section 5.

✏️ **Your turn.** Before querying in 7.3, drop pool cells with `pct_mito` above a threshold (a standard QC gate), then
re-run. Does uncertainty sampling stop wasting its budget? This one-line filter is what practitioners actually do.

### 📄 Research connection — Active learning and autonomous experimentation

- **What the field does.** Active learning (Settles 2009 is the canonical survey) asks a model to choose its own training labels — by uncertainty,
by expected model change, or by query-by-committee disagreement — to learn more per label. Its decision-theoretic
cousin is Thompson sampling (Russo *et al.* 2018). Taken to the lab, the same loop becomes **autonomous
experimentation**: King *et al.* (2009)'s *Robot Scientist* formed and tested hypotheses without a human in the loop,
and more recent systems close the design–build–test–learn cycle robotically.
- **What this notebook builds.** Section 7 is pool-based active learning with least-confidence querying and a logistic-regression classifier on
single-cell PCA features, plus the classic failure mode (uncertainty sampling chasing low-quality outliers) that
motivates density- and diversity-aware criteria. Production single-cell tools apply the same ideas to label transfer
and expert-in-the-loop annotation.
- **Where the frontier is.** As of 2025, "self-driving labs" that plan, run and interpret experiments — combining Bayesian optimisation, active
learning and, increasingly, language-model agents (Boiko *et al.* 2023; Module 43) — are a fast-moving and much-debated
frontier. Open problems include safe exploration, honest uncertainty out of distribution, reproducibility, and keeping
a human accountable for what an autonomous system proposes.
- **Read next.** Settles 2009, *Active Learning Literature Survey* (Univ. Wisconsin TR 1648); King *et al.* 2009 (*Science*), *The
automation of science*; Russo *et al.* 2018 (*Found Trends Mach Learn*), *A tutorial on Thompson sampling*.

---
## 8 · Pitfalls: where GPs and BO break

> 📍 **Where we are.** We provoke the four failures that bite GP/BO practitioners most, so you recognise them before they
> cost a month of experiments.

In [ ]:
# ── 8.1 The cubic wall, and a mis-specified kernel giving over-confident bands ──
sizes = [50, 100, 200, 400, 800]
times = []
for n in sizes:
    Xn = np.random.default_rng(0).uniform(0, 10, (n, 1))
    yn = enzyme_activity(Xn).ravel()
    t0 = time.time()
    for _ in range(3):
        K = rbf_kernel(Xn, Xn, 8.0, 0.5) + 0.01 * np.eye(n); cholesky(K, lower=True)
    times.append((time.time() - t0) / 3)
ratio = times[-1] / times[1]
print("GP fit time (Cholesky) scales ~O(n³):")
for n, t in zip(sizes, times):
    print(f"  n = {n:4d}: {t*1e3:6.1f} ms")
print(f"  8× more data ({sizes[1]}→{sizes[-1]}) cost ≈ {ratio:.0f}× more time — below the ~512× that pure O(n³) predicts,")
print("  because at these small n fixed per-call overhead and better BLAS efficiency at larger sizes both help; the")
print("  cubic scaling (and O(n²) memory for the kernel matrix) dominate and cap exact GPs at a few thousand points.")

# mis-specified kernel: fit a too-long length scale to a wiggly truth → over-confident
rng = np.random.default_rng(SEED + 30)
Xw = rng.uniform(0, 10, 12)[:, None]
yw = np.sin(Xw).ravel() * 1.0 + rng.normal(0, 0.05, 12)
Xt = rng.uniform(0, 10, 500)[:, None]; yt = np.sin(Xt).ravel()
m, v = gp_predict(Xw, yw, Xt, lambda A, B: rbf_kernel(A, B, length=6.0, signal=1.0), noise=0.05)
cov = np.mean(np.abs(yt - m) <= 1.96 * np.sqrt(v))
print(f"\nMis-specified (too-smooth) kernel on a wiggly truth: 95 % band really covers only {cov:.0%} — over-confident.")

🔍 **What just happened?** Two sobering facts. First, the **cubic cost**: fitting a GP needs a Cholesky factorisation of an
$n\times n$ matrix, so compute grows like $O(n^3)$ and memory like $O(n^2)$. A few thousand points is the comfortable
limit for an exact GP; beyond that you need approximations (inducing points, variational GPs — Section 9). Second,
**kernel mis-specification destroys calibration**: force a smooth (long length-scale) kernel onto a wiggly function and the
posterior becomes *over-confident* — its 95 % band covers far less than 95 %. The beautiful uncertainty of Section 2 is
only as trustworthy as the kernel and noise model behind it.

Two more pitfalls, without code because they are conceptual:

* **Non-stationarity.** A standard kernel assumes the function wiggles the same everywhere. Biology rarely obliges —
  activity may be flat across a stable regime and then fall off a denaturation cliff. A stationary GP smears the cliff and
  mis-sets its length scale. Fixes: input warping, non-stationary kernels, or a monotonic/mechanistic mean function.
* **Out-of-distribution exploitation.** This is the dangerous one for design. BO and MLDE *trust the surrogate to rank
  unseen points*. Where the surrogate has no data it may be confidently wrong, and the acquisition function will eagerly
  propose exactly those points ("the model says this never-tried variant is amazing"). Calibrated uncertainty is the
  defence — UCB and EI down-weight the unknown — but only if the uncertainty is honest, which out of distribution it often
  is not. [Module 39](39_generative_design.ipynb) builds designs that fall into this trap and the trust-region and
  pessimism fixes that climb back out. Real labs add a further reality check: batch effects and plate-position artefacts
  (Module 31) can masquerade as signal, so a well that "wins" may just have been in a lucky corner of the plate.

---
## 9 · 🧭 From toy to real data

Our GP fit a handful of points (sixteen for the enzyme curve) in milliseconds; the acquisition functions are a few lines. Real
machine-learning-guided experimentation keeps the ideas and changes the scale, the tooling and the failure modes.

**Scale and tooling.** Exact GPs handle up to a few thousand points; past that, production uses
[**GPyTorch**](https://gpytorch.ai) (GPU-accelerated, approximate GPs) and [**BoTorch**](https://botorch.org) (Monte-Carlo
acquisition, batch and multi-objective BO) built on PyTorch, or [scikit-learn's `GaussianProcessRegressor`](https://scikit-learn.org/stable/modules/gaussian_process.html)
for small problems. [**Ax**](https://ax.dev) wraps BoTorch for everyday experiment design. Hyper-parameter tuning —
BO's first mainstream hit — lives in Optuna, Ray Tune and the like.

**Public landscapes to practise on (reading, not downloads).** For proteins, [**ProteinGym**](https://proteingym.org)
(deep-mutational-scan benchmarks) and **FLIP** (fitness-landscape splits) let you test surrogates and acquisition rules on
real measured fitness. [**ChEMBL**](https://www.ebi.ac.uk/chembl/) and the [**Therapeutics Data Commons**](https://tdcommons.ai)
supply molecular-property tasks for BO over chemistry (Modules 39–40). GPs also power **spatial transcriptomics** — SpatialDE
and related tools model expression as a GP over tissue coordinates to find spatially variable genes (Module 33) — and
classic GP time-series models.

**Self-driving labs.** Closing the design–build–test–learn loop with robotics turns BO and active learning into
autonomous experimentation (King *et al.* 2009; Boiko *et al.* 2023; see [Module 43](43_llms_ai_agents.ipynb) on
agents). The promise is faster discovery; the responsibilities — safe exploration, honest uncertainty, reproducibility,
and human accountability — scale with it.

**The first three things that go wrong.** (1) **The objective is not what you measured** — assay noise, batch and
plate-position effects (Module 31), and a proxy readout that drifts from the real goal. (2) **Uncertainty is wrong out of
distribution**, so BO exploits surrogate errors (Section 8; Module 39). (3) **The kernel/representation mismatches the
biology** — epistasis, non-stationarity, or the wrong sequence/molecule encoding — so the surrogate ranks the wrong
candidates highest. Budget experiments to *test* the model's confident predictions, not only to chase its optimum.

---
## 📌 Key takeaways

- **A GP is Bayesian linear regression with infinitely many features.** Choosing a **kernel** (RBF, Matérn, linear) is
  choosing how smooth you believe the unknown function is; the posterior over functions comes in closed form from one
  Cholesky factorisation — a mean **and** a calibrated variance that grows away from the data.
- **The uncertainty is honest if the model is right.** Our 95 % band covered fresh noisy assays ≈ 95 % of the time (and
  the smooth truth essentially always — conservative, the safe side); a mis-specified (too-smooth) kernel made it badly
  over-confident. Always add the noise variance for a *measurement* interval.
- **The marginal likelihood picks hyper-parameters with Occam's razor built in** — it penalises over-flexible models — and
  it agreed with cross-validation on the length scale while needing no held-out split.
- **Bayesian optimisation turns uncertainty into decisions.** EI (derived from a truncated Gaussian) and UCB balance
  exploit vs explore; averaged over ten seeds both came within 1 % of the hidden culture optimum in ≈ 11 experiments and
  essentially reached it by 26, while random search and a coarse grid never got within 1 % in that budget.
- **Batches for 96-well plates** (Thompson sampling, constant liar) spread experiments across the surface; one parallel
  plate trades a little efficiency per experiment for a huge saving in *rounds*.
- **MLDE: a GP surrogate + UCB found a top variant from 0.3 % of a 160,000-variant library**, crushing random mutagenesis.
  On this *mostly additive* landscape the simple additive kernel matched the flexible RBF — more flexibility is not
  automatically better (bias–variance again) — but an additive model's accuracy is **capped** by the epistatic fraction,
  which only an interaction-aware surrogate can exceed.
- **Active learning** spends a labelling budget on the most uncertain samples and learns a little faster when classes
  overlap — but pure uncertainty **chases outliers** (it wasted much of its budget on damaged cells); QC filtering and
  density/diversity-aware querying fix that.
- **The recurring danger is out-of-distribution over-confidence**: acquisition functions exploit surrogate errors where
  there are no data. Calibrated uncertainty is the defence, and [Module 39](39_generative_design.ipynb) shows it failing
  and the fixes.

---
## ✅ Checkpoint

1. In one sentence, why does a GP's posterior variance grow far from the training points but a ridge-regression prediction's does not?
   <details><summary>Answer</summary>The GP keeps the full Bayesian posterior over functions, whose variance $K_{\star\star}-K_\star^\top(K+\sigma^2I)^{-1}K_\star$ is barely reduced where no nearby data constrain it; ridge returns a single point estimate (a MAP weight vector) and, by default, no predictive variance at all.</details>
2. What does the RBF length scale $\ell$ mean *biologically* for enzyme activity vs temperature, and what goes wrong if you set it far too small?
   <details><summary>Answer</summary>ℓ is the temperature range over which activity can change appreciably — how 'correlated' nearby temperatures are. Too small an ℓ lets the mean chase every noisy point (overfitting) and makes the uncertainty band collapse too fast (over-confidence / poor coverage).</details>
3. Compute a GP posterior by hand. With a *linear* kernel $k(x,x')=xx'$, noise $\sigma^2=0$, and two observations $f(1)=2,\,f(2)=4$, what is the posterior mean at $x_\star=3$? Why?
   <details><summary>Answer</summary>The data lie exactly on $f(x)=2x$, and a linear-kernel GP is Bayesian linear regression through the origin — the kernel $k=xx'$ has **no** bias term — so the posterior mean is $2\times 3 = 6$. (A caveat: with $\sigma^2=0$ the Gram matrix $[[1,2],[2,4]]$ is singular — both points sit on one line through the origin — so in practice you add a tiny jitter; the mean is unchanged.)</details>
4. Why does **expected improvement** explore, even though it is defined relative to the current best value?
   <details><summary>Answer</summary>EI = $(\mu-f^+)\Phi(z)+\sigma\phi(z)$ has a second term proportional to the posterior σ: a point with a mediocre mean but large uncertainty can still have high EI, because it *might* turn out far above $f^+$. So EI is large where the mean is high **or** where we are ignorant.</details>
5. Why can't an **additive** (linear-kernel) surrogate capture epistasis, while an **RBF on the one-hot sequence** can at least partly?
   <details><summary>Answer</summary>A linear kernel gives $k=\sum_\text{sites}$(matches), so the model is a sum of independent per-site contributions — no interaction terms. The RBF, $k=\exp(-(4-\text{matches})/\ell)$, is a *nonlinear* function of the joint number of matching sites, so two variants are 'similar' only if they agree on the right *combinations* — it can represent interactions, though not arbitrary ones.</details>
6. Your 96-well batch chosen by plain EI contains 96 copies of nearly the same well. What happened, and name one fix.
   <details><summary>Answer</summary>Plain EI is a single-point rule; its maximiser does not change until you add a real observation, so greedily taking the top-96 EI points stacks them on one spot. Fixes: Thompson sampling (one posterior draw per well) or the constant-liar trick (insert a fixed constant for each pick, re-fit, pick again).</details>
7. In Section 7 a pool polluted with damaged cells made **uncertainty sampling** waste most of its budget. What happened, and when else does least-confidence querying backfire?
   <details><summary>Answer</summary>Damaged outlier cells lie far from every class, so they look maximally *uncertain* and least-confidence sampling keeps querying them — it chases ambiguity, which is not the same as informativeness (it wasted up to ~88 % of early queries on damaged cells vs the 12 % base rate). It also backfires on rare classes the model becomes wrongly confident about (they stop looking uncertain and are never queried). Fixes: QC-filter the pool first, and add a density/diversity (representativeness) term.</details>
8. Code reading: in `run_bo`, the line `acq[chosen] = -np.inf` appears before `np.argmax(acq)`. What would break if it were removed?
   <details><summary>Answer</summary>Without it the acquisition could pick an already-measured candidate (its EI/UCB may still be high), so the loop would re-run the same experiment and make no progress. Masking measured points forces each step to choose a *new* experiment.</details>

## 🧠 Questions & discussion

1. Bayesian optimisation trusts the GP's uncertainty to decide where to look. If the uncertainty is wrong out of distribution (Section 8), BO can waste a whole campaign chasing a phantom optimum. How would you design a lab protocol that spends *some* budget deliberately testing the model's confident predictions, not only its optimum?
2. EI, UCB and Thompson sampling all balance exploration and exploitation differently. For a project where a failed experiment is cheap but a missed optimum is catastrophic, which would you pick, and how would you set its exploration knob?
3. The marginal likelihood chooses hyper-parameters without a held-out set, which is a gift when data are scarce — but it assumes the kernel family is correct. When would you *distrust* the marginal likelihood and fall back on cross-validation ([Module 23](23_model_selection.ipynb)) or a different kernel?
4. Self-driving labs let an algorithm choose and run experiments autonomously. Who is accountable when an autonomous campaign proposes an unsafe or wasteful experiment, and what guard-rails (human sign-off, safe-exploration constraints, logging) would you require before letting a loop run unattended?
5. Active learning saved labels on average but squandered much of its budget on damaged outlier cells. In a clinical annotation project where expert time is the scarce resource, how would you change the acquisition rule to avoid chasing junk, and how would you *measure* that the change worked without peeking at the full labels?

## 🚀 Bonus — going further

- Implement **GP classification** with a Laplace or variational approximation (the likelihood is no longer Gaussian), and use it to drive active learning in Section 7 directly from the GP's class probabilities.
- Build **multi-objective BO**: optimise enzyme *activity* and *stability* at once by targeting the Pareto front (expected hypervolume improvement). Compare with optimising a fixed weighted sum.
- Add **heteroscedastic noise** (assay noise that grows with the signal) by making $\sigma$ a second GP, and check whether the predictive intervals re-calibrate.
- Swap the RBF for a **Matérn-5/2** kernel throughout and re-run the BO and MLDE experiments. Does the rougher prior help or hurt on these landscapes, and does the marginal likelihood prefer it?
- Re-run Section 6 with a **random-forest ensemble** surrogate (uncertainty = tree disagreement) instead of a GP, and compare its MLDE learning curve and its out-of-distribution calibration with the GP's.

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module: 29 · Gaussian processes, Bayesian optimisation and active learning
Biological question: which experiment to run next to reach the best result in the fewest trials, with honest uncertainty
Data (simulated): enzyme_activity(T) curve + culture_yield(temp, pH) surface (hidden optima, defined in-notebook);
                  4-site Potts landscape from simulate_helix_bundle + simulate_protein_family (true h, J = oracle);
                  simulate_scrna PBMC cells (known types) for active learning
Method: from-scratch GP (Cholesky posterior); marginal-likelihood hyper-parameters; EI/PI/UCB acquisition; batch BO
        (Thompson, constant liar); GP-surrogate MLDE (RBF vs linear sequence kernel, UCB); pool active learning
Key equation: f̄⋆ = Kₛᵀ(K+σ²I)⁻¹y; var = K⋆⋆ − KₛᵀK⁻¹Kₛ; EI = (μ−f⁺)Φ(z)+σφ(z)
Key parameters: length scale ℓ, signal, noise σ; EI's ξ; UCB's β; batch size; labelling budget
Ground-truth check result: 95 % band coverage ≈ ___ %; BO best yield vs random/grid after 26 experiments = ___;
                           MLDE best true fitness epistatic/additive/random = ___ (optimum ___); active-learning
                           accuracy vs random = ___
What changed when I modified … (kernel length / UCB β / greedy vs UCB surrogate / QC-filtering the pool): …
One failure mode: out-of-distribution over-confidence; mis-specified kernel kills coverage; uncertainty sampling
                  wastes the budget on outliers
Connection to research: Rasmussen & Williams 2006; Jones 1998 / Srinivas 2010 / Snoek 2012 (BO); Romero 2013 / Wu 2019
                        (MLDE); Settles 2009 / King 2009 (active learning, autonomous labs)
```

## 📚 References

- Rasmussen C. E., Williams C. K. I. (2006) *Gaussian Processes for Machine Learning*. MIT Press (free online).
- Jones D. R., Schonlau M., Welch W. J. (1998) Efficient global optimization of expensive black-box functions. *J Glob Optim* 13:455–492 (EGO / expected improvement).
- Srinivas N., Krause A., Kakade S., Seeger M. (2010) Gaussian process optimization in the bandit setting: no regret and experimental design. *ICML* (GP-UCB).
- Snoek J., Larochelle H., Adams R. P. (2012) Practical Bayesian optimization of machine learning algorithms. *NeurIPS*.
- Shahriari B., Swersky K., Wang Z., Adams R. P., de Freitas N. (2016) Taking the human out of the loop: a review of Bayesian optimization. *Proc IEEE* 104:148–175.
- Ginsbourger D., Le Riche R., Carraro L. (2010) Kriging is well-suited to parallelize optimization. In *Computational Intelligence in Expensive Optimization Problems* (the 'constant liar').
- Russo D., Van Roy B., Kazerouni A., Osband I., Wen Z. (2018) A tutorial on Thompson sampling. *Found Trends Mach Learn* 11:1–96.
- Romero P. A., Krause A., Arnold F. H. (2013) Navigating the protein fitness landscape with Gaussian processes. *PNAS* 110:E193–E201.
- Wu N. C., Dai L., Olson C. A., Lloyd-Smith J. O., Sun R. (2016) Adaptation in protein fitness landscapes is facilitated by indirect paths. *eLife* 5:e16965 (the measured GB1 four-site landscape).
- Wu Z., Kan S. B. J., Lewis R. D., Wittmann B. J., Arnold F. H. (2019) Machine learning-assisted directed protein evolution with combinatorial libraries. *PNAS* 116:8852–8858.
- Yang K. K., Wu Z., Arnold F. H. (2019) Machine-learning-guided directed evolution for protein engineering. *Nat Methods* 16:687–694.
- Hie B. L., Yang K. K. (2022) Adaptive machine learning for protein engineering. *Curr Opin Struct Biol* 72:145–152.
- Settles B. (2009) *Active Learning Literature Survey*. Univ. Wisconsin–Madison CS Tech Rep 1648.
- King R. D. *et al.* (2009) The automation of science. *Science* 324:85–89 (the Robot Scientist).
- Boiko D. A., MacKnight R., Kline B., Gomes G. (2023) Autonomous chemical research with large language models. *Nature* 624:570–578.

## ➡️ Next

We now have the two skills that drive modern experimental ML — a surrogate with calibrated uncertainty, and a rule for
choosing the next experiment — and we have seen the danger that defines the generative-design module: acquisition
functions exploit the surrogate's errors where it has no data. **[Module 30 · Single-cell RNA-seq](30_single_cell_rnaseq.ipynb)**
opens Part G and turns to the cell; the GP and active-learning ideas return later in the course — GPs model spatially
variable genes in **[Module 33 · Spatial omics](33_spatial_omics.ipynb)** — while **[Module 39 · Generative
design](39_generative_design.ipynb)** takes the protein-engineering loop to its limit, provoking out-of-distribution
failure on purpose and fixing it with trust regions and pessimism.

In [ ]:
# ── Total run time ──
print(f"⏱ Notebook compute: {time.time() - T_START:.0f} s on CPU.")